# LSB eHealth — Complete Base Papers Reproduction — Google Colab

This notebook recreates the **two base-paper reconstructions only**:

1. **DBH-Chain documented reconstruction**
2. **EPP-BCFL documented reconstruction**

It does **not** run the proposed LSB eHealth method.

## Locked comparison rule
Final LSB claims must be compared against the **measured reconstruction outputs generated under the available compute environment**, not against published headline numbers. Published values may be exported separately as context only.

## Recovered old notebook lineage
- `01_DBH_Chain_Reproduction_Batched.ipynb`
- `01_DBH_Chain_Reproduction_Batch1_Final.ipynb`
- `02_EPP_BCFL_Reproduction_Batched.ipynb`
- `02_EPP_BCFL_PAPER_ALIGNED_PILOT_50R_v6.ipynb`
- combined workflow: `LSB_BASE_REPRODUCE_THEN_LSB_SAME_DATASETS_FIXED.ipynb`

## Default resource-aligned run
- DBH: seeds 42,43,44 × node counts 20,50,100,500,1000.
- EPP: 10 clients, seed 42, 50 rounds, 2 local epochs, 5 classes/client, WRN-16-4 GroupNorm.
- EPP additional base metrics: ε=0.5 DP, data poisoning, model poisoning, persistent malicious-client FGSM, supplementary raw white-box FGSM, security response time, and device heterogeneity.
- Google Drive checkpointing after every FL round.
- Re-running **Run All** resumes completed work and does not restart finished runs.

> The papers do not disclose enough executable detail for bit-for-bit reproduction. Outputs are therefore labelled **reconstruction**.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [3]:

# 0 — Colab + Drive bootstrap
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import os, json, time, shutil, zipfile

MASTER_ROOT = Path('/content/drive/MyDrive/LSB_EHEALTH_COMPLETE_BASE_PAPERS_COLAB')
MASTER_ROOT.mkdir(parents=True, exist_ok=True)

print('MASTER ROOT:', MASTER_ROOT, flush=True)
print('This notebook contains BASE PAPER reconstructions only.', flush=True)


Mounted at /content/drive
MASTER ROOT: /content/drive/MyDrive/LSB_EHEALTH_COMPLETE_BASE_PAPERS_COLAB
This notebook contains BASE PAPER reconstructions only.


# Part A — DBH-Chain reconstruction

The DBH track preserves the earlier documented-reconstruction logic: public paper-listed Kaggle sources are loaded when resolvable; CMS/HealthData remain explicitly unresolved unless an exact file/URL is supplied. MMPS arrivals, queue/service delays, Knapsack scheduling, PoV reconstruction, tamper validation, energy, consensus messages and latency are measured.

All DBH seeds/configurations persist directly to Google Drive.


In [4]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("kagglehub") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","kagglehub"])
print("Dependencies ready.", flush=True)

Dependencies ready.


In [5]:
from __future__ import annotations
import dataclasses, hashlib, json, math, os, random, shutil, time, warnings, zipfile
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import kagglehub
warnings.filterwarnings("default")

def progress(msg): print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:

# A0 — DBH configuration (resource-aligned reconstruction)
PROJECT_ID="DBH_CHAIN_REPRO_COLAB_V1"
SEED_PLAN="CORE_3"
SEEDS=(42,43,44)
SAVE_TASK_LEVEL_DETAIL=False

DBH_NODE_COUNTS=(20,50,100,500,1000)
DBH_SECONDS=3600
MMPS_TRANSITION=((0.90,0.10),(0.20,0.80))
MMPS_RATES=(8.0,25.0)
FOG_SERVICE_RATE=25.0
CLOUD_SERVICE_RATE=45.0
NETWORK_DELAY_MS=(8.0,25.0)
DEADLINE_MS=200.0
RESOURCE_CAPACITY=60
TAMPER_FRACTION=0.05
BLOCKCHAIN_BYZANTINE_FRACTION=0.20
SIMULATED_NETWORK_RTT_MS=5.0
SIMULATED_MESSAGE_PROCESSING_MS=0.01
SIMULATED_HASH_VERIFICATION_MS=0.05

DBH_KAGGLE_HANDLES=(
    "palwashakhanzadi/iot-blockchain-data",
    "kanakbaghel/hospital-management-dataset",
    "prasad22/healthcare-dataset",
)

# Leave blank unless you have the exact verified files/URLs.
CMS_HOSPITAL_EXACT_URL_OR_PATH=""
HEALTHDATA_SERVICE_EXACT_URL_OR_PATH=""
STRICT_ALL_FIVE_SOURCES=False

ROOT=MASTER_ROOT/"DBH_CHAIN_REPRO_V1"
DIRS={n:ROOT/n for n in ("datasets","checkpoints","models","metrics","plots","logs","manifests")}
for p in DIRS.values():
    p.mkdir(parents=True,exist_ok=True)

print('DBH ROOT:', ROOT, flush=True)


DBH ROOT: /content/drive/MyDrive/LSB_EHEALTH_COMPLETE_BASE_PAPERS_COLAB/DBH_CHAIN_REPRO_V1


In [7]:
SCIENTIFIC_CONFIG={
 "node_counts":DBH_NODE_COUNTS,"seconds":DBH_SECONDS,"seeds":SEEDS,
 "mmps_transition":MMPS_TRANSITION,"mmps_rates":MMPS_RATES,
 "fog_rate":FOG_SERVICE_RATE,"cloud_rate":CLOUD_SERVICE_RATE,
 "network_delay_ms":NETWORK_DELAY_MS,"deadline_ms":DEADLINE_MS,
 "resource_capacity":RESOURCE_CAPACITY,"tamper_fraction":TAMPER_FRACTION,
 "byzantine_fraction":BLOCKCHAIN_BYZANTINE_FRACTION,
 "implementation":"v4-consensus-aware-validation-DBH-only",
}
EXPERIMENT_FINGERPRINT=hashlib.sha256(json.dumps(SCIENTIFIC_CONFIG,sort_keys=True,default=str).encode()).hexdigest()[:16]


manifest_cfg=DIRS['manifests']/ 'scientific_config.json'
if manifest_cfg.exists():
    old_fp=json.loads(manifest_cfg.read_text()).get('fingerprint')
    if old_fp and old_fp!=EXPERIMENT_FINGERPRINT:
        raise RuntimeError(
            f'DBH fingerprint mismatch existing={old_fp} active={EXPERIMENT_FINGERPRINT}. '
            'Use a fresh Drive folder for a different scientific configuration.'
        )
manifest_cfg.write_text(json.dumps({
    'project_id':PROJECT_ID,
    'fingerprint':EXPERIMENT_FINGERPRINT,
    'config':SCIENTIFIC_CONFIG
},indent=2,default=str))
print('DBH experiment fingerprint:',EXPERIMENT_FINGERPRINT,flush=True)

def write_resume_bundle():
    bundle=MASTER_ROOT/'DBH_CHAIN_REPRO_V1_Resume.zip'
    tmp=bundle.with_suffix('.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
        for folder_name in ('checkpoints','models','metrics','manifests','logs','plots'):
            folder=DIRS[folder_name]
            for f in folder.rglob('*'):
                if f.is_file():
                    z.write(f,f.relative_to(ROOT))
    tmp.replace(bundle)
    return bundle


DBH experiment fingerprint: 92ff6911ad08557b


In [8]:
@dataclass
class DBHConfig:
    dbh_kaggle_handles: tuple[str,...]=DBH_KAGGLE_HANDLES
    cms_hospital_exact_url_or_path: str=CMS_HOSPITAL_EXACT_URL_OR_PATH
    healthdata_service_exact_url_or_path: str=HEALTHDATA_SERVICE_EXACT_URL_OR_PATH
    strict_dbh_all_five_sources: bool=STRICT_ALL_FIVE_SOURCES
    allow_documented_reconstruction: bool=True
    deadline_ms: float=DEADLINE_MS
    mmps_transition: tuple=MMPS_TRANSITION
    mmps_rates: tuple=MMPS_RATES
    dbh_seconds: int=DBH_SECONDS
    resource_capacity: int=RESOURCE_CAPACITY
    tamper_fraction: float=TAMPER_FRACTION
    committee_size: int=21
    low_battery: float=0.20
    fog_service_rate: float=FOG_SERVICE_RATE
    cloud_service_rate: float=CLOUD_SERVICE_RATE
    network_delay_ms: tuple=NETWORK_DELAY_MS
    blockchain_byzantine_fraction: float=BLOCKCHAIN_BYZANTINE_FRACTION
    simulated_network_rtt_ms: float=SIMULATED_NETWORK_RTT_MS
    simulated_message_processing_ms: float=SIMULATED_MESSAGE_PROCESSING_MS
    simulated_hash_verification_ms: float=SIMULATED_HASH_VERIFICATION_MS
CFG=DBHConfig()

ASSUMPTIONS=pd.DataFrame([
 ["IoT Blockchain Data",DBH_KAGGLE_HANDLES[0],"PAPER_SUPPORTED"],
 ["Hospital Management Dataset",DBH_KAGGLE_HANDLES[1],"PAPER_SUPPORTED"],
 ["CMS Hospital Dataset",CMS_HOSPITAL_EXACT_URL_OR_PATH or "UNRESOLVED","UNRESOLVED_SOURCE"],
 ["HealthData Hospital Service Dataset",HEALTHDATA_SERVICE_EXACT_URL_OR_PATH or "UNRESOLVED","UNRESOLVED_SOURCE"],
 ["Kaggle Healthcare Dataset",DBH_KAGGLE_HANDLES[2],"PAPER_SUPPORTED"],
 ["MMPS/service parameters","paper omits exact values used here","DOCUMENTED_RECONSTRUCTION"],
],columns=["item","value","status"])
ASSUMPTIONS.to_csv(DIRS['manifests']/ 'assumptions.csv',index=False)
display(ASSUMPTIONS)

,item,value,status
0,IoT Blockchain Data,palwashakhanzadi/iot-blockchain-data,PAPER_SUPPORTED
1,Hospital Management Dataset,kanakbaghel/hospital-management-dataset,PAPER_SUPPORTED
2,CMS Hospital Dataset,UNRESOLVED,UNRESOLVED_SOURCE
3,HealthData Hospital Service Dataset,UNRESOLVED,UNRESOLVED_SOURCE
4,Kaggle Healthcare Dataset,prasad22/healthcare-dataset,PAPER_SUPPORTED
5,MMPS/service parameters,paper omits exact values used here,DOCUMENTED_RECONSTRUCTION


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
# A1–A2 Load, fetch, profile and inventory all DBH paper sources
import urllib.request

DBH_TABLES = {}
DBH_DATASET_REGISTRY = []

def sha256_file(path: Path, chunk_size: int = 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def read_structured_tables(root: Path):
    tables = {}
    candidates = [root] if root.is_file() else sorted(root.rglob("*"))
    for file in candidates:
        if not file.is_file():
            continue
        try:
            suffix = file.suffix.lower()
            if suffix == ".csv":
                tables[file.name] = pd.read_csv(file)
            elif suffix == ".parquet":
                tables[file.name] = pd.read_parquet(file)
            elif suffix in {".json", ".jsonl"}:
                tables[file.name] = pd.read_json(file, lines=suffix == ".jsonl")
            elif suffix in {".xlsx", ".xls"}:
                tables[file.name] = pd.read_excel(file)
        except Exception as exc:
            print("Skipped unreadable structured file:", file, type(exc).__name__, exc)
    return tables

def resolve_exact_url_or_path(name: str, value: str):
    record = {"name": name, "source": value, "status": "UNRESOLVED", "path": "", "error": ""}
    if not value:
        record["error"] = (
            "The paper does not identify an exact file. Supply a verified direct URL "
            "or attach the exact file and set its Kaggle path."
        )
        return record, {}
    try:
        candidate = Path(value)
        if candidate.exists():
            local = candidate
        elif value.startswith(("https://", "http://")):
            suffix = Path(value.split("?")[0]).suffix or ".csv"
            local = DIRS["datasets"] / f"{name.lower().replace(' ', '_')}{suffix}"
            urllib.request.urlretrieve(value, local)
        else:
            raise FileNotFoundError(value)
        record.update({"status": "READY", "path": str(local)})
        return record, read_structured_tables(local)
    except Exception as exc:
        record["status"] = "FAILED"
        record["error"] = f"{type(exc).__name__}: {exc}"
        return record, {}

def find_attached_dataset(handle: str):
    # Optional local mirror in Drive. Otherwise kagglehub downloads the public dataset.
    slug = handle.split("/")[-1]
    local_root = MASTER_ROOT / "DBH_LOCAL_DATA"
    if local_root.exists():
        exact = [p for p in local_root.rglob(slug) if p.is_dir()]
        if exact:
            return exact[0]
    return None

if True:
    for handle in CFG.dbh_kaggle_handles:
        record = {"name": handle.split("/")[-1], "source": handle, "status": "PENDING", "path": "", "error": ""}
        try:
            attached = find_attached_dataset(handle)
            path = attached if attached is not None else Path(kagglehub.dataset_download(handle))
            record.update({"status": "READY", "path": str(path)})
            DBH_TABLES[handle] = read_structured_tables(path)
        except Exception as exc:
            record.update({"status": "FAILED", "error": f"{type(exc).__name__}: {exc}"})
        DBH_DATASET_REGISTRY.append(record)

    for name, value in (
        ("CMS Hospital Dataset", CFG.cms_hospital_exact_url_or_path),
        ("HealthData Hospital Service Dataset", CFG.healthdata_service_exact_url_or_path),
    ):
        record, tables = resolve_exact_url_or_path(name, value)
        DBH_DATASET_REGISTRY.append(record)
        if tables:
            DBH_TABLES[name] = tables

    registry = pd.DataFrame(DBH_DATASET_REGISTRY)
    registry.to_csv(DIRS["manifests"] / "dbh_dataset_registry.csv", index=False)

    # Deterministic cleaning:
    # - remove exact duplicate rows;
    # - remove rows where every field is missing;
    # - do not blindly impute missing clinical values.
    file_rows = []
    cleaned_tables = {}
    for source, tables in DBH_TABLES.items():
        cleaned_tables[source] = {}
        for table_name, frame in tables.items():
            before_rows = len(frame)
            before_duplicates = int(frame.duplicated().sum())
            before_missing = int(frame.isna().sum().sum())

            cleaned = (
                frame.dropna(how="all")
                .drop_duplicates()
                .reset_index(drop=True)
            )
            cleaned_tables[source][table_name] = cleaned

            file_rows.append({
                "source": source,
                "table": table_name,
                "rows_before": before_rows,
                "rows_after": len(cleaned),
                "columns": len(cleaned.columns),
                "duplicates_removed": before_duplicates,
                "missing_cells_before": before_missing,
                "missing_cells_after": int(cleaned.isna().sum().sum()),
                "all_missing_rows_removed": int(
                    before_rows
                    - len(frame.dropna(how="all"))
                ),
                "missing_value_policy": "preserved_and_logged_no_blind_imputation",
            })

    DBH_TABLES = cleaned_tables
    profiling = pd.DataFrame(file_rows)
    profiling.to_csv(
        DIRS["metrics"] / "dbh_dataset_cleaning_profile.csv",
        index=False
    )

    unresolved = registry[registry["status"] != "READY"]
    if CFG.strict_dbh_all_five_sources and not unresolved.empty:
        raise RuntimeError(
            "Strict five-source mode is enabled, but these exact DBH sources are unresolved:\n"
            + unresolved[["name", "error"]].to_string(index=False)
        )
    if not CFG.allow_documented_reconstruction:
        reconstruction_rows = ASSUMPTIONS[
            ASSUMPTIONS["status"] == "DOCUMENTED_RECONSTRUCTION"
        ]
        if not reconstruction_rows.empty:
            raise RuntimeError(
                "Documented reconstruction is disabled, but the papers omit required settings."
            )

    display(registry)
    display(profiling)

Using Colab cache for faster access to the 'iot-blockchain-data' dataset.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Using Colab cache for faster access to the 'hospital-management-dataset' dataset.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Using Colab cache for faster access to the 'healthcare-dataset' dataset.


,name,source,status,path,error
0,iot-blockchain-data,palwashakhanzadi/iot-blockchain-data,READY,/kaggle/input/iot-blockchain-data,
1,hospital-management-dataset,kanakbaghel/hospital-management-dataset,READY,/kaggle/input/hospital-management-dataset,
2,healthcare-dataset,prasad22/healthcare-dataset,READY,/kaggle/input/healthcare-dataset,
3,CMS Hospital Dataset,,UNRESOLVED,,The paper does not identify an exact file. Sup...
4,HealthData Hospital Service Dataset,,UNRESOLVED,,The paper does not identify an exact file. Sup...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,source,table,rows_before,rows_after,columns,duplicates_removed,missing_cells_before,missing_cells_after,all_missing_rows_removed,missing_value_policy
0,palwashakhanzadi/iot-blockchain-data,account.csv,780,276,5,503,575,0,115,preserved_and_logged_no_blind_imputation
1,palwashakhanzadi/iot-blockchain-data,order.csv,665,133,6,532,0,0,0,preserved_and_logged_no_blind_imputation
2,palwashakhanzadi/iot-blockchain-data,transaction.csv,278,278,8,0,0,0,0,preserved_and_logged_no_blind_imputation
3,kanakbaghel/hospital-management-dataset,appointments.csv,200,200,7,0,0,0,0,preserved_and_logged_no_blind_imputation
4,kanakbaghel/hospital-management-dataset,billing.csv,200,200,7,0,0,0,0,preserved_and_logged_no_blind_imputation
5,kanakbaghel/hospital-management-dataset,doctors.csv,10,10,8,0,0,0,0,preserved_and_logged_no_blind_imputation
6,kanakbaghel/hospital-management-dataset,patients.csv,50,50,11,0,0,0,0,preserved_and_logged_no_blind_imputation
7,kanakbaghel/hospital-management-dataset,treatments.csv,200,200,6,0,0,0,0,preserved_and_logged_no_blind_imputation
8,prasad22/healthcare-dataset,healthcare_dataset.csv,55500,54966,15,534,0,0,0,preserved_and_logged_no_blind_imputation


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
# A3–A8 Dataset-to-task mapping, MMPS, queue, Knapsack and consensus validation

PRIORITY_MAP = {
    "critical": 10.0,
    "life threatening": 10.0,
    "emergency": 9.0,
    "urgent": 8.0,
    "severe": 8.0,
    "serious": 7.0,
    "high": 7.0,
    "moderate": 5.0,
    "medium": 5.0,
    "stable": 4.0,
    "low": 3.0,
    "routine": 2.0,
    "normal": 2.0,
}

@dataclass
class HealthcareTask:
    task_id: int
    arrival_time: float
    payload_bytes: int
    priority: float
    resource_requirement: int
    deadline_ms: float
    source: str
    expected_digest: str
    observed_digest: str
    tampered: bool = False


def parse_priority(value):
    """Auditable priority parser; unknown categorical values use neutral priority 5."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return 5.0, "missing_neutral"

    if isinstance(value, (int, float, np.integer, np.floating)):
        number = float(value)
        if not np.isfinite(number):
            return 5.0, "nonfinite_neutral"
        if 0.0 <= number <= 1.0:
            return 1.0 + 9.0 * number, "numeric_unit_interval"
        return float(np.clip(number, 1.0, 10.0)), "numeric_clipped"

    text = str(value).strip().lower()
    for label, score in PRIORITY_MAP.items():
        if label in text:
            return score, f"mapped:{label}"
    return 5.0, "unknown_categorical_neutral"


def source_rows_to_templates(tables):
    templates = []
    mapping_rows = []
    task_id = 0

    for dataset_name, group in tables.items():
        for table_name, frame in group.items():
            if frame.empty:
                continue

            priority_columns = [
                column for column in frame.columns
                if any(token in column.lower() for token in (
                    "priority", "urgent", "severity", "condition", "status"
                ))
            ]

            for _, row in frame.head(5000).iterrows():
                raw = row.to_json(date_format="iso", default_handler=str)
                priority_value = (
                    row[priority_columns[0]]
                    if priority_columns else None
                )
                priority, mapping_rule = parse_priority(priority_value)
                digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()

                templates.append(HealthcareTask(
                    task_id=task_id,
                    arrival_time=0.0,
                    payload_bytes=len(raw.encode("utf-8")),
                    priority=priority,
                    resource_requirement=max(1, int(row.notna().sum())),
                    deadline_ms=CFG.deadline_ms,
                    source=f"{dataset_name}:{table_name}",
                    expected_digest=digest,
                    observed_digest=digest,
                ))
                mapping_rows.append({
                    "source": f"{dataset_name}:{table_name}",
                    "task_id": task_id,
                    "priority_column": priority_columns[0] if priority_columns else "",
                    "priority_rule": mapping_rule,
                    "priority": priority,
                    "payload_bytes": len(raw.encode("utf-8")),
                    "resource_requirement": max(1, int(row.notna().sum())),
                })
                task_id += 1

    mapping_audit = pd.DataFrame(mapping_rows)
    mapping_audit.to_csv(
        DIRS["metrics"] / "dbh_dataset_to_task_mapping_audit.csv",
        index=False
    )
    return templates


class MMPSGenerator:
    def __init__(self, transition, rates, seed):
        self.transition = np.asarray(transition, dtype=float)
        self.rates = np.asarray(rates, dtype=float)
        if (
            self.transition.shape != (len(rates), len(rates))
            or not np.allclose(self.transition.sum(axis=1), 1.0)
        ):
            raise ValueError("Invalid MMPS transition matrix")
        self.rng = np.random.default_rng(seed)
        self.state = 0

    def arrivals(self, seconds):
        output = []
        for second in range(seconds):
            count = int(self.rng.poisson(self.rates[self.state]))
            output.extend(second + self.rng.random(count))
            self.state = int(
                self.rng.choice(
                    len(self.rates),
                    p=self.transition[self.state]
                )
            )
        return sorted(output)


def knapsack_select(tasks, capacity):
    if not tasks or capacity <= 0:
        return []
    n = len(tasks)
    dp = np.zeros((n + 1, capacity + 1), dtype=float)

    for i, task in enumerate(tasks, start=1):
        weight = min(capacity, max(1, int(task.resource_requirement)))
        dp[i] = dp[i - 1]
        candidate = (
            dp[i - 1, : capacity + 1 - weight]
            + float(task.priority)
        )
        dp[i, weight:] = np.maximum(dp[i, weight:], candidate)

    selected = []
    remaining = capacity
    for i in range(n, 0, -1):
        if dp[i, remaining] > dp[i - 1, remaining] + 1e-12:
            task = tasks[i - 1]
            selected.append(task)
            remaining -= min(
                capacity,
                max(1, int(task.resource_requirement))
            )
    return list(reversed(selected))


def build_workload(templates, seed):
    if not templates:
        raise RuntimeError(
            "No DBH structured rows were loaded. Enable Kaggle Internet "
            "or attach the paper-listed datasets."
        )

    rng = np.random.default_rng(seed)
    arrivals = MMPSGenerator(
        CFG.mmps_transition,
        CFG.mmps_rates,
        seed
    ).arrivals(CFG.dbh_seconds)

    tasks = []
    for task_id, arrival in enumerate(arrivals):
        template = templates[task_id % len(templates)]
        tampered = bool(rng.random() < CFG.tamper_fraction)
        observed_digest = template.expected_digest
        if tampered:
            observed_digest = hashlib.sha256(
                f"tampered:{template.expected_digest}:{task_id}".encode()
            ).hexdigest()

        tasks.append(dataclasses.replace(
            template,
            task_id=task_id,
            arrival_time=float(arrival),
            tampered=tampered,
            observed_digest=observed_digest,
        ))
    return tasks


def blockchain_energy(payload_bytes, messages, cpu_ms):
    return (
        payload_bytes * 5e-7
        + messages * 2e-4
        + cpu_ms * 2e-5
    )


def consensus_parameters(system, node_count):
    if system == "DBH_POV_RECONSTRUCTION":
        validators = min(
            node_count,
            max(4, int(math.ceil(math.log2(max(node_count, 2)))))
        )
        tolerated = (validators - 1) // 2
        threshold = validators // 2 + 1
        messages = 2 * validators
        phases = 1
    elif system == "LSB_ENERGY_VRF_PBFT":
        validators = min(CFG.committee_size, node_count)
        tolerated = (validators - 1) // 3
        threshold = 2 * tolerated + 1
        messages = 2 * validators * max(0, validators - 1)
        phases = 3
    else:
        raise ValueError(system)

    latency_ms = (
        phases * CFG.simulated_network_rtt_ms
        + messages * CFG.simulated_message_processing_ms
        + CFG.simulated_hash_verification_ms
    )
    return {
        "validators": validators,
        "tolerated": tolerated,
        "threshold": threshold,
        "messages": messages,
        "latency_ms": latency_ms,
    }


def validate_transaction(task, consensus, rng):
    actual_valid = task.expected_digest == task.observed_digest

    validators = consensus["validators"]
    byzantine_count = min(
        validators,
        int(round(validators * CFG.blockchain_byzantine_fraction))
    )
    byzantine_ids = set(
        rng.choice(
            validators,
            size=byzantine_count,
            replace=False
        ).tolist()
    ) if byzantine_count else set()

    votes = []
    for validator_id in range(validators):
        honest_vote = actual_valid
        vote = (
            not honest_vote
            if validator_id in byzantine_ids
            else honest_vote
        )
        votes.append(vote)

    predicted_valid = sum(votes) >= consensus["threshold"]
    return {
        "ground_truth_valid": actual_valid,
        "predicted_valid": predicted_valid,
        "validation_correct": predicted_valid == actual_valid,
        "yes_votes": int(sum(votes)),
        "selected_byzantine": byzantine_count,
    }


def run_blockchain_workload(tasks, node_count, seed, system):
    rng = np.random.default_rng(seed)
    rows = []
    batch_size = max(1, int(np.mean(CFG.mmps_rates)))
    consensus = consensus_parameters(system, node_count)
    consensus_lane_available_ms = 0.0

    for start in range(0, len(tasks), batch_size):
        batch = tasks[start:start + batch_size]
        selected_tasks = knapsack_select(batch, CFG.resource_capacity)
        selected_ids = {task.task_id for task in selected_tasks}
        used_capacity = sum(
            task.resource_requirement for task in selected_tasks
        )
        utilization = min(
            1.0,
            used_capacity / max(CFG.resource_capacity, 1)
        )

        preliminary = []
        for queue_position, task in enumerate(batch):
            if task.task_id not in selected_ids:
                rows.append({
                    "system": system,
                    "node_count": node_count,
                    "task_id": task.task_id,
                    "arrival_time": task.arrival_time,
                    "scheduled": False,
                    "tampered": task.tampered,
                    "ground_truth_valid": not task.tampered,
                    "predicted_valid": np.nan,
                    "validation_correct": np.nan,
                    "committed": False,
                    "hybrid_delay_ms": np.inf,
                    "deadline_met": False,
                    "energy_j": 0.0,
                    "consensus_messages": 0,
                    "resource_utilization": utilization,
                    "commit_time_ms": np.nan,
                })
                continue

            layer = "fog" if task.priority >= 5 else "cloud"
            service_rate = (
                CFG.fog_service_rate
                if layer == "fog" else CFG.cloud_service_rate
            )
            queue_ms = (
                queue_position / max(service_rate, 1e-9) * 1000
            )
            service_ms = (
                task.resource_requirement
                / max(service_rate, 1e-9)
                * 1000
            )
            scale = (
                1.0
                + math.log2(max(node_count, 2)) / 20.0
            )
            communication_ms = float(
                rng.uniform(*CFG.network_delay_ms)
            ) * scale
            validation = validate_transaction(task, consensus, rng)
            ready_ms = (
                task.arrival_time * 1000
                + queue_ms
                + service_ms
                + communication_ms
            )
            preliminary.append({
                "task": task,
                "layer": layer,
                "queue_ms": queue_ms,
                "service_ms": service_ms,
                "communication_ms": communication_ms,
                "ready_ms": ready_ms,
                "validation": validation,
                "resource_utilization": utilization,
            })

        if not preliminary:
            continue

        block_start_ms = max(
            consensus_lane_available_ms,
            max(item["ready_ms"] for item in preliminary)
        )
        block_finish_ms = (
            block_start_ms + consensus["latency_ms"]
        )
        consensus_lane_available_ms = block_finish_ms

        for item in preliminary:
            task = item["task"]
            validation = item["validation"]
            storage_ms = float(rng.uniform(2.0, 8.0))
            commit_time_ms = block_finish_ms + storage_ms
            hybrid_ms = (
                commit_time_ms - task.arrival_time * 1000
            )
            committed = bool(validation["predicted_valid"])

            rows.append({
                "system": system,
                "node_count": node_count,
                "task_id": task.task_id,
                "arrival_time": task.arrival_time,
                "scheduled": True,
                "tampered": task.tampered,
                **validation,
                "committed": committed,
                "layer": item["layer"],
                "queue_delay_ms": item["queue_ms"],
                "service_delay_ms": item["service_ms"],
                "communication_delay_ms": item["communication_ms"],
                "consensus_delay_ms": consensus["latency_ms"],
                "storage_delay_ms": storage_ms,
                "hybrid_delay_ms": hybrid_ms,
                "deadline_met": (
                    committed and hybrid_ms <= task.deadline_ms
                ),
                "energy_j": blockchain_energy(
                    task.payload_bytes,
                    consensus["messages"],
                    item["service_ms"] + consensus["latency_ms"],
                ),
                "consensus_messages": consensus["messages"],
                "resource_utilization": item["resource_utilization"],
                "commit_time_ms": commit_time_ms if committed else np.nan,
                "committee_size": consensus["validators"],
                "byzantine_tolerance": consensus["tolerated"],
            })

    return pd.DataFrame(rows)


def safe_rate(numerator, denominator):
    return numerator / denominator if denominator else np.nan


def summarize_blockchain(frame, seed):
    scheduled = frame[frame["scheduled"]].copy()
    committed = frame[frame["committed"]].copy()
    eligible = frame[~frame["tampered"]].copy()
    completed_eligible = eligible[eligible["committed"]]
    finite = scheduled[np.isfinite(scheduled["hybrid_delay_ms"])]

    valid_scheduled = scheduled[
        scheduled["ground_truth_valid"] == True
    ]
    invalid_scheduled = scheduled[
        scheduled["ground_truth_valid"] == False
    ]

    last_commit_ms = (
        committed["commit_time_ms"].max()
        if not committed.empty else CFG.dbh_seconds * 1000
    )
    effective_seconds = max(
        CFG.dbh_seconds,
        float(last_commit_ms) / 1000
    )

    true_positive = int(
        (
            (invalid_scheduled["predicted_valid"] == False)
        ).sum()
    )
    false_negative = int(
        (
            (invalid_scheduled["predicted_valid"] == True)
        ).sum()
    )
    false_positive = int(
        (
            (valid_scheduled["predicted_valid"] == False)
        ).sum()
    )

    return {
        "system": frame["system"].iloc[0],
        "node_count": int(frame["node_count"].iloc[0]),
        "seed": seed,
        "submitted": len(frame),
        "eligible_tasks": len(eligible),
        "scheduled": int(frame["scheduled"].sum()),
        "committed": int(frame["committed"].sum()),
        "throughput_tx_s": len(committed) / max(effective_seconds, 1e-9),
        "effective_measurement_seconds": effective_seconds,
        "mean_hybrid_delay_ms": finite["hybrid_delay_ms"].mean(),
        "p95_hybrid_delay_ms": finite["hybrid_delay_ms"].quantile(0.95),
        "scheduling_efficiency_pct": (
            safe_rate(len(completed_eligible), len(eligible)) * 100
        ),
        "scheduler_acceptance_pct": (
            safe_rate(int(frame["scheduled"].sum()), len(frame)) * 100
        ),
        "validation_integrity_pct": (
            scheduled["validation_correct"].mean() * 100
            if not scheduled.empty else np.nan
        ),
        "tamper_detection_recall": safe_rate(
            true_positive,
            true_positive + false_negative,
        ),
        "validation_false_positive_rate": safe_rate(
            false_positive,
            len(valid_scheduled),
        ),
        "invalid_acceptance_rate": safe_rate(
            false_negative,
            len(invalid_scheduled),
        ),
        "deadline_miss_pct": (
            (1 - completed_eligible["deadline_met"].mean()) * 100
            if not completed_eligible.empty else np.nan
        ),
        "mean_resource_utilization": (
            finite["resource_utilization"].mean()
            if not finite.empty else np.nan
        ),
        "energy_per_committed_task_j": (
            committed["energy_j"].mean()
            if not committed.empty else np.nan
        ),
        "mean_consensus_messages": (
            finite["consensus_messages"].mean()
            if not finite.empty else np.nan
        ),
        "mean_consensus_latency_ms": (
            finite["consensus_delay_ms"].mean()
            if not finite.empty else np.nan
        ),
    }

In [11]:

# A9 — Run every pending DBH configuration; save after each node/seed
templates=source_rows_to_templates(DBH_TABLES)

summary_path=DIRS['metrics']/ 'dbh_all_summary.csv'
if summary_path.exists():
    try:
        all_df=pd.read_csv(summary_path)
    except Exception:
        all_df=pd.DataFrame()
else:
    all_df=pd.DataFrame()

completed=set()
if not all_df.empty and {'system','node_count','seed'}.issubset(all_df.columns):
    completed=set(
        (str(r.system),int(r.node_count),int(r.seed))
        for r in all_df.itertuples(index=False)
    )

new_rows=[]
total_expected=len(SEEDS)*len(DBH_NODE_COUNTS)
for seed in SEEDS:
    progress(f'DBH SEED START → {seed}')
    workload=build_workload(templates,seed)
    for nodes in DBH_NODE_COUNTS:
        key=('DBH_POV_RECONSTRUCTION',int(nodes),int(seed))
        if key in completed:
            progress(f'  SKIP completed node_count={nodes} seed={seed}')
            continue
        progress(f'  RUN node_count={nodes} seed={seed}')
        result=run_blockchain_workload(workload,nodes,seed,'DBH_POV_RECONSTRUCTION')
        result['seed']=seed
        row=summarize_blockchain(result,seed)
        new_rows.append(row)

        if SAVE_TASK_LEVEL_DETAIL:
            result.to_csv(DIRS['metrics']/f'dbh_detail_nodes{nodes}_seed{seed}.csv',index=False)

        base = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
        merged = pd.concat([base,pd.DataFrame([row])],ignore_index=True)
        merged = merged.drop_duplicates(['system','node_count','seed'],keep='last')
        merged.to_csv(summary_path,index=False)
        write_resume_bundle()
        completed.add(key)
    progress(f'DBH SEED COMPLETE → {seed}')

all_df=pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame(new_rows)
print('Completed DBH configurations:',len(all_df),'/',total_expected,flush=True)
display(all_df.sort_values(['seed','node_count']) if not all_df.empty else all_df)


[06:38:36] DBH SEED START → 42


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[06:38:36]   SKIP completed node_count=20 seed=42
[06:38:36]   SKIP completed node_count=50 seed=42
[06:38:36]   SKIP completed node_count=100 seed=42
[06:38:36]   SKIP completed node_count=500 seed=42
[06:38:36]   SKIP completed node_count=1000 seed=42
[06:38:36] DBH SEED COMPLETE → 42
[06:38:36] DBH SEED START → 43
[06:38:37]   SKIP completed node_count=20 seed=43
[06:38:37]   SKIP completed node_count=50 seed=43
[06:38:37]   SKIP completed node_count=100 seed=43
[06:38:37]   SKIP completed node_count=500 seed=43
[06:38:37]   SKIP completed node_count=1000 seed=43
[06:38:37] DBH SEED COMPLETE → 43
[06:38:37] DBH SEED START → 44
[06:38:37]   SKIP completed node_count=20 seed=44
[06:38:37]   SKIP completed node_count=50 seed=44
[06:38:37]   SKIP completed node_count=100 seed=44
[06:38:37]   SKIP completed node_count=500 seed=44
[06:38:37]   SKIP completed node_count=1000 seed=44
[06:38:37] DBH SEED COMPLETE → 44
Completed DBH configurations: 15 / 15


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,system,node_count,seed,submitted,eligible_tasks,scheduled,committed,throughput_tx_s,effective_measurement_seconds,mean_hybrid_delay_ms,...,scheduler_acceptance_pct,validation_integrity_pct,tamper_detection_recall,validation_false_positive_rate,invalid_acceptance_rate,deadline_miss_pct,mean_resource_utilization,energy_per_committed_task_j,mean_consensus_messages,mean_consensus_latency_ms
0,DBH_POV_RECONSTRUCTION,20,42,49896,47407,15866,15069,4.185597,3600.203365,888.866097,...,31.798140,100.0,1.0,0.0,0.0,100.0,0.988113,0.011603,10.0,5.15
1,DBH_POV_RECONSTRUCTION,50,42,49896,47407,15866,15069,4.185595,3600.205035,889.971393,...,31.798140,100.0,1.0,0.0,0.0,100.0,0.988113,0.012003,12.0,5.17
2,DBH_POV_RECONSTRUCTION,100,42,49896,47407,15866,15069,4.185593,3600.206303,890.812391,...,31.798140,100.0,1.0,0.0,0.0,100.0,0.988113,0.012403,14.0,5.19
3,DBH_POV_RECONSTRUCTION,500,42,49896,47407,15866,15069,4.185608,3600.193778,892.826412,...,31.798140,100.0,1.0,0.0,0.0,100.0,0.988113,0.013204,18.0,5.23
4,DBH_POV_RECONSTRUCTION,1000,42,49896,47407,15866,15069,4.185607,3600.194541,893.668667,...,31.798140,100.0,1.0,0.0,0.0,100.0,0.988113,0.013605,20.0,5.25
5,DBH_POV_RECONSTRUCTION,20,43,50320,47799,15970,15147,4.207500,3600.000000,878.045153,...,31.736884,100.0,1.0,0.0,0.0,100.0,0.988190,0.011617,10.0,5.15
6,DBH_POV_RECONSTRUCTION,50,43,50320,47799,15970,15147,4.207500,3600.000000,879.155532,...,31.736884,100.0,1.0,0.0,0.0,100.0,0.988190,0.012017,12.0,5.17
7,DBH_POV_RECONSTRUCTION,100,43,50320,47799,15970,15147,4.207500,3600.000000,880.000372,...,31.736884,100.0,1.0,0.0,0.0,100.0,0.988190,0.012418,14.0,5.19
8,DBH_POV_RECONSTRUCTION,500,43,50320,47799,15970,15147,4.207500,3600.000000,882.101292,...,31.736884,100.0,1.0,0.0,0.0,100.0,0.988190,0.013218,18.0,5.23
9,DBH_POV_RECONSTRUCTION,1000,43,50320,47799,15970,15147,4.207500,3600.000000,882.949968,...,31.736884,100.0,1.0,0.0,0.0,100.0,0.988190,0.013619,20.0,5.25


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
# Context-only paper reference; NOT the primary LSB comparison target.
PUBLISHED_DBH=pd.DataFrame([
 ["Scheduling efficiency (%)",99.32,"reported up to"],
 ["Hybrid delay reduction (%)","41–57","reported range"],
 ["Validation integrity (%)",98.87,"reported up to"],
],columns=["metric","paper_value","note"])
PUBLISHED_DBH.to_csv(DIRS['metrics']/ 'dbh_published_reference.csv',index=False)
display(PUBLISHED_DBH)
if (DIRS['metrics']/ 'dbh_all_summary.csv').exists():
    x=pd.read_csv(DIRS['metrics']/ 'dbh_all_summary.csv')
    if not x.empty:
        display(x.groupby('node_count')[["throughput_tx_s","mean_hybrid_delay_ms","scheduling_efficiency_pct","validation_integrity_pct"]].agg(['mean','std']))

,metric,paper_value,note
0,Scheduling efficiency (%),99.32,reported up to
1,Hybrid delay reduction (%),41–57,reported range
2,Validation integrity (%),98.87,reported up to


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


throughput_tx_s           mean_hybrid_delay_ms            \
                      mean       std                 mean       std   
node_count                                                            
20                4.172624  0.042860           885.991967  6.969383   
50                4.172623  0.042861           887.098977  6.966507   
100               4.172623  0.042861           887.941271  6.964333   
500               4.172627  0.042864           889.977597  6.907517   
1000              4.172626  0.042865           890.821922  6.903779   

           scheduling_efficiency_pct           validation_integrity_pct       
                                mean       std                     mean  std  
node_count                                                                    
20                         31.808007  0.131176                    100.0  0.0  
50                         31.808007  0.131176                    100.0  0.0  
100                        31.808007  0.131176                    100.0  0.0  
500                        31.808007  0.131176                    100.0  0.0  
1000                       31.808007  0.131176                    100.0  0.0

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:

# A10 — DBH reconstruction benchmark export
dbh_summary_path=DIRS['metrics']/ 'dbh_all_summary.csv'
if dbh_summary_path.exists():
    dbh_measured=pd.read_csv(dbh_summary_path)
    dbh_benchmark=dbh_measured.groupby('node_count',as_index=False).agg(
        throughput_tx_s_mean=('throughput_tx_s','mean'),
        mean_hybrid_delay_ms_mean=('mean_hybrid_delay_ms','mean'),
        p95_hybrid_delay_ms_mean=('p95_hybrid_delay_ms','mean'),
        scheduling_efficiency_pct_mean=('scheduling_efficiency_pct','mean'),
        validation_integrity_pct_mean=('validation_integrity_pct','mean'),
        energy_per_committed_task_j_mean=('energy_per_committed_task_j','mean'),
        mean_consensus_messages_mean=('mean_consensus_messages','mean'),
        mean_consensus_latency_ms_mean=('mean_consensus_latency_ms','mean'),
        deadline_miss_pct_mean=('deadline_miss_pct','mean'),
    )
    dbh_benchmark.to_csv(DIRS['metrics']/ 'dbh_reproduced_benchmark.csv',index=False)
    display(dbh_benchmark)
    print('✅ DBH reconstruction benchmark saved.',flush=True)


,node_count,throughput_tx_s_mean,mean_hybrid_delay_ms_mean,p95_hybrid_delay_ms_mean,scheduling_efficiency_pct_mean,validation_integrity_pct_mean,energy_per_committed_task_j_mean,mean_consensus_messages_mean,mean_consensus_latency_ms_mean,deadline_miss_pct_mean
0,20,4.172624,885.991967,1380.216214,31.808007,100.0,0.011593,10.0,5.15,100.0
1,50,4.172623,887.098977,1381.365661,31.808007,100.0,0.011994,12.0,5.17,100.0
2,100,4.172623,887.941271,1382.124327,31.808007,100.0,0.012394,14.0,5.19,100.0
3,500,4.172627,889.977597,1384.148916,31.808007,100.0,0.013195,18.0,5.23,100.0
4,1000,4.172626,890.821922,1385.149089,31.808007,100.0,0.013595,20.0,5.25,100.0


✅ DBH reconstruction benchmark saved.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Part B — EPP-BCFL reconstruction

Default **RESOURCE_ALIGNED** profile recreates the earlier 10-client / seed-42 / 50-round baseline used on the available compute environment and adds the missing comparison experiments:

- CLEAN
- DP ε = 0.5
- 20% malicious-client data poisoning
- 20% malicious-client model poisoning
- 20% malicious-client **FGSM during local training** for 50 rounds
- supplementary raw white-box FGSM
- security response time (operational reconstruction = anomaly-detection time + consensus-verification latency)
- device/network heterogeneity

The clean, poisoning, model-poisoning, FGSM-training and DP runs each have independent round checkpoints in Google Drive.


In [14]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('kagglehub') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','kagglehub'])
print('Dependencies ready.',flush=True)

Dependencies ready.


In [15]:
from __future__ import annotations
import copy, hashlib, json, math, os, random, shutil, time, warnings, zipfile
from collections import OrderedDict
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
warnings.filterwarnings('default')

def progress(msg): print(f"[{time.strftime('%H:%M:%S')}] {msg}",flush=True)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [16]:

# B0 — EPP resource-aligned control panel
PROJECT_ID='EPP_BCFL_REPRO_COLAB_V1'
PROFILE='RESOURCE_ALIGNED'

SEEDS=(42,)
CLIENT_GRID=(10,)
REPRESENTATIVE_CLIENTS=10

GLOBAL_ROUNDS=50
LOCAL_EPOCHS=2
CLIENT_FRACTION=1.0
CLASSES_PER_CLIENT=5

LOCAL_BATCH_SIZE=64
LEARNING_RATE=0.05
MIN_LEARNING_RATE_FACTOR=0.02
MOMENTUM=0.9
WEIGHT_DECAY=5e-4

EARLY_STOP_PATIENCE=8
EARLY_STOP_MIN_ROUNDS=30
BEST_MIN_DELTA=1e-4

TARGET_EPSILONS=(0.5,)
DELTA=1e-5
CLIENT_CLIP_NORM=25.0
CLIP_CLEAN_UPDATES=False

MAX_AGG_WEIGHT_FACTOR=2.0
ADVERSARIAL_CLIENT_FRACTION=0.20
DATA_POISON_FRACTION=0.30
MODEL_POISON_SCALE=-5.0
FGSM_EPSILON=8/255
ANOMALY_MAD_THRESHOLD=3.5
POS_BFT_BYZANTINE_FRACTION=0.20
VALIDATION_FRACTION=0.10
NUM_WORKERS=0

# Drive-safe checkpoint cadence.
RESUME_EVERY_ROUNDS=10

ROOT=MASTER_ROOT/'EPP_BCFL_REPRO_V4'
DIRS={n:ROOT/n for n in ('datasets','checkpoints','models','metrics','plots','logs','manifests')}
for p in DIRS.values():
    p.mkdir(parents=True,exist_ok=True)

print('EPP ROOT:',ROOT,flush=True)


EPP ROOT: /content/drive/MyDrive/LSB_EHEALTH_COMPLETE_BASE_PAPERS_COLAB/EPP_BCFL_REPRO_V4


In [17]:
SCIENTIFIC_CONFIG={
 'project_id':PROJECT_ID,
 'seed_plan':SEED_PLAN,'seeds':SEEDS,'client_grid':CLIENT_GRID,
 'representative_clients':REPRESENTATIVE_CLIENTS,
 'global_rounds':GLOBAL_ROUNDS,'local_epochs':LOCAL_EPOCHS,
 'client_fraction':CLIENT_FRACTION,
 'classes_per_client':CLASSES_PER_CLIENT,
 'local_batch_size':LOCAL_BATCH_SIZE,
 'optimizer':['SGD',LEARNING_RATE,MOMENTUM,WEIGHT_DECAY,'cosine',MIN_LEARNING_RATE_FACTOR],
 'validation_fraction':VALIDATION_FRACTION,
 'architecture':'WideResNet-16-4 GroupNorm reconstruction',
 'best_checkpoint':['validation_accuracy',EARLY_STOP_PATIENCE,EARLY_STOP_MIN_ROUNDS,BEST_MIN_DELTA],
 'clean_clipping':CLIP_CLEAN_UPDATES,
 'dp':[TARGET_EPSILONS,DELTA,CLIENT_CLIP_NORM],
 'attacks':[ADVERSARIAL_CLIENT_FRACTION,DATA_POISON_FRACTION,MODEL_POISON_SCALE,FGSM_EPSILON],
}
EXPERIMENT_FINGERPRINT=hashlib.sha256(
    json.dumps(SCIENTIFIC_CONFIG,sort_keys=True,default=str).encode()
).hexdigest()[:16]


manifest_cfg=DIRS['manifests']/ 'scientific_config.json'
if manifest_cfg.exists():
    old=json.loads(manifest_cfg.read_text())
    old_fp=old.get('fingerprint')
    if old_fp and old_fp!=EXPERIMENT_FINGERPRINT:
        raise RuntimeError(
            f'EPP fingerprint mismatch existing={old_fp} active={EXPERIMENT_FINGERPRINT}. '
            'Use a fresh Drive folder for a changed scientific configuration.'
        )
manifest_cfg.write_text(json.dumps({
    'project_id':PROJECT_ID,
    'fingerprint':EXPERIMENT_FINGERPRINT,
    'config':SCIENTIFIC_CONFIG
},indent=2,default=str))
print('EPP experiment fingerprint:',EXPERIMENT_FINGERPRINT,flush=True)


EPP experiment fingerprint: fc6eb52bf20bc86d


In [18]:
def detect_device():
    if not torch.cuda.is_available(): return torch.device('cpu'),{'reason':'CUDA unavailable'}
    try:
        major,minor=torch.cuda.get_device_capability(0); arch=f'sm_{major}{minor}'; supported=list(torch.cuda.get_arch_list())
        if supported and arch not in supported: return torch.device('cpu'),{'gpu':torch.cuda.get_device_name(0),'reason':f'{arch} unsupported'}
        x=torch.ones(1,device='cuda')+1; torch.cuda.synchronize(); del x
        return torch.device('cuda'),{'gpu':torch.cuda.get_device_name(0),'arch':arch}
    except Exception as exc: return torch.device('cpu'),{'reason':str(exc)}
DEVICE,DEVICE_DETAILS=detect_device(); CUDA_USABLE=DEVICE.type=='cuda'
print('Selected device:',DEVICE,DEVICE_DETAILS,flush=True)
if not CUDA_USABLE: raise RuntimeError('Use a compatible T4/L4 GPU for this batched EPP notebook.')

def seed_everything(seed):
    os.environ['PYTHONHASHSEED']=str(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
seed_everything(SEEDS[0])

Selected device: cuda {'gpu': 'Tesla T4', 'arch': 'sm_75'}


In [19]:
@dataclass
class EPPConfig:
    seed: int=42
    force_rerun: bool=False
    num_workers: int=NUM_WORKERS
    learning_rate: float=LEARNING_RATE
    min_learning_rate_factor: float=MIN_LEARNING_RATE_FACTOR
    momentum: float=MOMENTUM
    weight_decay: float=WEIGHT_DECAY
    local_batch_size: int=LOCAL_BATCH_SIZE
    local_epochs: int=LOCAL_EPOCHS
    global_rounds: int=GLOBAL_ROUNDS
    client_fraction: float=CLIENT_FRACTION
    non_iid_method: str='LIMITED_CLASSES'
    classes_per_client: int=CLASSES_PER_CLIENT
    max_aggregation_weight_factor: float=MAX_AGG_WEIGHT_FACTOR
    delta: float=DELTA
    client_clip_norm: float=CLIENT_CLIP_NORM
    clip_clean_updates: bool=CLIP_CLEAN_UPDATES
    early_stop_patience: int=EARLY_STOP_PATIENCE
    early_stop_min_rounds: int=EARLY_STOP_MIN_ROUNDS
    best_min_delta: float=BEST_MIN_DELTA
    adversarial_client_fraction: float=ADVERSARIAL_CLIENT_FRACTION
    data_poison_fraction: float=DATA_POISON_FRACTION
    model_poison_scale: float=MODEL_POISON_SCALE
    anomaly_mad_threshold: float=ANOMALY_MAD_THRESHOLD
    pos_bft_byzantine_fraction: float=POS_BFT_BYZANTINE_FRACTION
    simulated_network_rtt_ms: float=5.0
    simulated_message_processing_ms: float=0.01

CFG=EPPConfig()
CFG.client_grid=CLIENT_GRID

def stage_checkpoint(name):
    return DIRS['checkpoints']/f'{name}.pt'

def stage_csv(name):
    return DIRS['metrics']/f'{name}.csv'

def validate_checkpoint_fingerprint(state,name):
    if state.get('fingerprint')!=EXPERIMENT_FINGERPRINT:
        raise RuntimeError(f'Checkpoint fingerprint mismatch: {name}')

def load_pipeline_checkpoint(path,map_location='cpu'):
    p=Path(path).resolve()
    if ROOT.resolve() not in p.parents and p!=ROOT.resolve():
        raise RuntimeError(f'Unsafe checkpoint path {p}')
    progress(f'CHECKPOINT LOAD → {p.name}')
    try:
        return torch.load(p,map_location=map_location,weights_only=False)
    except TypeError:
        return torch.load(p,map_location=map_location)

def write_resume_bundle():
    bundle=MASTER_ROOT/'EPP_BCFL_REPRO_V4_Resume.zip'
    tmp=bundle.with_suffix('.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
        for folder_name in ('checkpoints','models','metrics','manifests','logs','plots'):
            for f in DIRS[folder_name].rglob('*'):
                if f.is_file():
                    z.write(f,f.relative_to(ROOT))
    tmp.replace(bundle)
    return bundle

def write_slim_results_bundle():
    bundle=MASTER_ROOT/'EPP_BCFL_REPRO_V4_RESULTS_SLIM.zip'
    tmp=bundle.with_suffix('.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
        for folder_name in ('metrics','manifests','logs','plots'):
            for f in DIRS[folder_name].rglob('*'):
                if f.is_file():
                    z.write(f,f.relative_to(ROOT))
        for f in DIRS['models'].glob('epp_clean_n*_seed*.pt'):
            z.write(f,f.relative_to(ROOT))
    tmp.replace(bundle)
    return bundle

def maybe_write_resume_bundle(round_number):
    if round_number%RESUME_EVERY_ROUNDS==0 or round_number==GLOBAL_ROUNDS:
        p=write_resume_bundle()
        progress(f'RESUME SNAPSHOT → {p} | {p.stat().st_size/1024**2:.1f} MB')

print(
    'EPP V4 config validated | '
    f'batch={CFG.local_batch_size} | classes/client={CFG.classes_per_client} | '
    f'clean_clip={CFG.clip_clean_updates} | best-checkpoint=ON',
    flush=True,
)


EPP V4 config validated | batch=64 | classes/client=5 | clean_clip=False | best-checkpoint=ON


In [20]:
# Context-only paper reference; measured reconstruction remains primary.
PUBLISHED_EPP=pd.DataFrame([
 [10,88.5,87.2,85.6,86.4,20],
 [20,90.1,88.9,87.5,88.2,18],
 [30,92.3,90.5,89.8,90.2,15],
 [40,93.8,92.1,91.4,91.7,12],
 [50,95.2,94.0,93.3,93.6,10],
],columns=['num_clients','paper_accuracy','paper_precision','paper_recall','paper_f1','paper_convergence_epochs'])
PUBLISHED_EPP.to_csv(DIRS['metrics']/ 'epp_published_reference.csv',index=False)
display(PUBLISHED_EPP)

,num_clients,paper_accuracy,paper_precision,paper_recall,paper_f1,paper_convergence_epochs
0,10,88.5,87.2,85.6,86.4,20
1,20,90.1,88.9,87.5,88.2,18
2,30,92.3,90.5,89.8,90.2,15
3,40,93.8,92.1,91.4,91.7,12
4,50,95.2,94.0,93.3,93.6,10


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [21]:
# Reuse an attached CIFAR cache if available, otherwise torchvision downloads it.
def detect_cifar_root():
    root=MASTER_ROOT/'datasets'/'cifar10'
    root.mkdir(parents=True,exist_ok=True)
    return str(root)
CIFAR_ROOT=detect_cifar_root()
MEAN=(0.4914,0.4822,0.4465); STD=(0.2470,0.2435,0.2616)
train_transform=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
eval_transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
train_aug=datasets.CIFAR10(CIFAR_ROOT,train=True,download=True,transform=train_transform)
train_eval=datasets.CIFAR10(CIFAR_ROOT,train=True,download=True,transform=eval_transform)
test_eval=datasets.CIFAR10(CIFAR_ROOT,train=False,download=True,transform=eval_transform)
all_idx=np.arange(len(train_eval))
all_labels=np.asarray(train_eval.targets)
train_indices,val_indices=train_test_split(
    all_idx,
    test_size=VALIDATION_FRACTION,
    random_state=42,
    stratify=all_labels,
)
train_indices=np.asarray(train_indices,dtype=int)
val_indices=np.asarray(val_indices,dtype=int)
test_indices=np.arange(len(test_eval))
VALSET=Subset(train_eval,list(map(int,val_indices)))
TESTSET=Subset(test_eval,list(map(int,test_indices)))
print({
    'official_train':len(train_eval),
    'federated_train':len(train_indices),
    'validation':len(val_indices),
    'official_test':len(test_eval),
    'note':'validation is used for best-round selection; official test remains untouched',
},flush=True)


100%|██████████| 170M/170M [37:05<00:00, 76.6kB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/datasets/utils.py:213: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(to_path)


{'official_train': 50000, 'federated_train': 45000, 'validation': 5000, 'official_test': 10000, 'note': 'validation is used for best-round selection; official test remains untouched'}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [22]:
# Shared reconstruction CNN: CIFAR WideResNet-16-4 with GroupNorm.
# The paper states CNN but does not disclose its exact architecture.

class WideBasic(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, groups=16):
        super().__init__()
        g1 = min(groups, in_channels)
        while in_channels % g1 != 0 and g1 > 1:
            g1 -= 1
        g2 = min(groups, out_channels)
        while out_channels % g2 != 0 and g2 > 1:
            g2 -= 1

        self.norm1 = nn.GroupNorm(g1, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(g2, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.shortcut = (
            nn.Identity()
            if stride == 1 and in_channels == out_channels
            else nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False)
        )

    def forward(self, x):
        out = F.silu(self.norm1(x))
        shortcut = self.shortcut(out if not isinstance(self.shortcut, nn.Identity) else x)
        out = self.conv1(out)
        out = self.conv2(F.silu(self.norm2(out)))
        return out + shortcut


class CifarWideResNet16x4(nn.Module):
    def __init__(self, classes=10):
        super().__init__()
        self.stem = nn.Conv2d(3, 16, 3, padding=1, bias=False)

        # WRN-16-4: 2 residual blocks per group, widths 64/128/256.
        self.layer1 = nn.Sequential(
            WideBasic(16, 64, 1),
            WideBasic(64, 64, 1),
        )
        self.layer2 = nn.Sequential(
            WideBasic(64, 128, 2),
            WideBasic(128, 128, 1),
        )
        self.layer3 = nn.Sequential(
            WideBasic(128, 256, 2),
            WideBasic(256, 256, 1),
        )

        self.norm = nn.GroupNorm(16, 256)
        self.fc = nn.Linear(256, classes)

        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='linear')
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = F.silu(self.norm(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)


def make_model():
    return CifarWideResNet16x4().to(DEVICE)


def finite_model(model):
    return all(
        not torch.is_floating_point(v) or torch.isfinite(v).all().item()
        for v in model.state_dict().values()
    )


def assert_finite_model(model, context="model"):
    bad = [
        k for k, v in model.state_dict().items()
        if torch.is_floating_point(v) and not torch.isfinite(v).all()
    ]
    if bad:
        raise FloatingPointError(f"{context} has non-finite tensors: {bad[:10]}")


@torch.no_grad()
def evaluate(model, dataset, adversarial_epsilon=0.0):
    model = model.to(DEVICE).eval()
    assert_finite_model(model, "evaluation model")
    loader = DataLoader(
        dataset,
        batch_size=256,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=CUDA_USABLE,
    )
    losses, truth, predicted = [], [], []

    for batch_id, (x, y) in enumerate(loader):
        x = x.to(DEVICE, non_blocking=CUDA_USABLE)
        y = y.to(DEVICE, non_blocking=CUDA_USABLE)

        if adversarial_epsilon > 0:
            with torch.enable_grad():
                x_adv = x.detach().clone().requires_grad_(True)
                loss_adv = F.cross_entropy(model(x_adv), y)
                gradient = torch.autograd.grad(loss_adv, x_adv)[0]
                std = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)
                mean = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
                epsilon_normalized = adversarial_epsilon / std
                lower = (0.0 - mean) / std
                upper = (1.0 - mean) / std
                x = torch.maximum(
                    torch.minimum(
                        x_adv.detach() + epsilon_normalized * gradient.sign(),
                        upper,
                    ),
                    lower,
                )

        logits = model(x)
        if not torch.isfinite(logits).all():
            raise FloatingPointError(f"Non-finite logits in evaluation batch {batch_id}")

        loss = F.cross_entropy(logits, y)
        losses.append(float(loss.item()))
        truth.extend(y.cpu().tolist())
        predicted.extend(logits.argmax(1).cpu().tolist())

    precision, recall, f1, _ = precision_recall_fscore_support(
        truth, predicted, average="macro", zero_division=0
    )
    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(truth, predicted),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": confusion_matrix(truth, predicted).tolist(),
    }

preview = make_model()
print('Model: WideResNet-16-4 GroupNorm')
print('Trainable parameters:', sum(p.numel() for p in preview.parameters() if p.requires_grad), flush=True)
del preview


Model: WideResNet-16-4 GroupNorm
Trainable parameters: 2748890


In [23]:
# B3–B4 Non-IID limited-class client partitions and heterogeneity audit

def client_partitions(n_clients: int, seed: int):
    if CFG.non_iid_method != "LIMITED_CLASSES":
        raise NotImplementedError(
            "This notebook locks the paper-aligned LIMITED_CLASSES reconstruction."
        )

    rng = np.random.default_rng(seed)
    indices = np.asarray(train_indices, dtype=int)
    labels = np.asarray(train_eval.targets)[indices]
    classes = np.unique(labels)

    # Assign exactly classes_per_client class slots per client while keeping
    # every class represented across the population.
    slots = np.resize(classes, n_clients * CFG.classes_per_client)
    rng.shuffle(slots)
    client_classes = {
        client: set(map(int, slots[
            client * CFG.classes_per_client:(client + 1) * CFG.classes_per_client
        ]))
        for client in range(n_clients)
    }

    # Guarantee unique class count when repeated slots land on the same class.
    for client in range(n_clients):
        while len(client_classes[client]) < CFG.classes_per_client:
            client_classes[client].add(int(rng.choice(classes)))

    output = {client: [] for client in range(n_clients)}
    for label in classes:
        class_indices = indices[labels == label].copy()
        rng.shuffle(class_indices)
        eligible = [c for c in range(n_clients) if int(label) in client_classes[c]]
        if not eligible:
            eligible = [int(label) % n_clients]
            client_classes[eligible[0]].add(int(label))
        for client, chunk in zip(eligible, np.array_split(class_indices, len(eligible))):
            output[client].extend(map(int, chunk))

    # A client may be empty only if the source sample is too small.
    empty = [c for c, values in output.items() if not values]
    if empty:
        raise RuntimeError(f"Empty clients after non-IID partition: {empty}")

    flat = [idx for values in output.values() for idx in values]
    if len(flat) != len(set(flat)) or set(flat) != set(map(int, train_indices)):
        raise AssertionError("Client partition duplicated or lost training samples.")

    audit_rows = []
    for client, values in output.items():
        local_labels = np.asarray(train_eval.targets)[values]
        audit_rows.append({
            "num_clients": n_clients,
            "seed": seed,
            "client_id": client,
            "samples": len(values),
            "class_count": len(np.unique(local_labels)),
            "classes": ",".join(map(str, sorted(np.unique(local_labels).tolist()))),
        })
    audit = pd.DataFrame(audit_rows)
    audit.to_csv(
        DIRS["metrics"] / f"client_partition_n{n_clients}_seed{seed}.csv",
        index=False
    )
    return output

# Visual audit for every client grid using the first seed.
partition_audit = []
for n_clients in CFG.client_grid:
    parts = client_partitions(n_clients, CFG.seed)
    for client, values in parts.items():
        local_labels = np.asarray(train_eval.targets)[values]
        partition_audit.append({
            "num_clients": n_clients,
            "client_id": client,
            "samples": len(values),
            "class_count": len(np.unique(local_labels)),
        })
PARTITION_AUDIT = pd.DataFrame(partition_audit)
PARTITION_AUDIT.to_csv(DIRS["metrics"] / "client_partition_audit.csv", index=False)
display(PARTITION_AUDIT.groupby("num_clients")[["samples", "class_count"]].describe())

samples                                                        \
              count    mean         std     min      25%     50%      75%   
num_clients                                                                 
10             10.0  4500.0  357.771187  3713.0  4312.25  4556.5  4799.75   

                    class_count                                     
                max       count mean  std  min  25%  50%  75%  max  
num_clients                                                         
10           4875.0        10.0  5.0  0.0  5.0  5.0  5.0  5.0  5.0

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [24]:
def blockchain_energy(payload_bytes,messages,cpu_ms):
    return payload_bytes*5e-7 + messages*2e-4 + cpu_ms*2e-5

In [25]:
# C2 Trainable-parameter updates, clipping and RDP-accounted aggregate noise

def trainable_keys(model):
    return set(dict(model.named_parameters()))


def state_delta(local_state, global_state, keys):
    output = OrderedDict((k, local_state[k].float() - global_state[k].float()) for k in keys)
    assert_finite_update(output, "raw update")
    return output


def assert_finite_update(update, context="update"):
    bad = [k for k, v in update.items() if not torch.isfinite(v).all()]
    if bad:
        raise FloatingPointError(f"{context} has non-finite tensors: {bad[:10]}")


def update_norm(update):
    value = math.sqrt(sum(float((v.float() ** 2).sum()) for v in update.values()))
    if not math.isfinite(value):
        raise FloatingPointError("Update norm is non-finite")
    return value


def clip_update(update, max_norm):
    norm = update_norm(update)
    factor = min(1.0, max_norm / (norm + 1e-12))
    clipped = OrderedDict((k, v.float() * factor) for k, v in update.items())
    return clipped, norm


def normalize_capped_weights(raw_weights):
    weights = np.asarray(raw_weights, dtype=np.float64)
    if len(weights) == 0 or not np.isfinite(weights).all() or np.any(weights < 0) or weights.sum() <= 0:
        raise ValueError("Invalid aggregation weights")
    weights = weights / weights.sum()
    cap = CFG.max_aggregation_weight_factor / len(weights)
    weights = np.minimum(weights, cap)
    weights = weights / weights.sum()
    return weights


def weighted_average(updates, raw_weights):
    weights = normalize_capped_weights(raw_weights)
    keys = list(updates[0])
    result = OrderedDict()
    for key in keys:
        accumulator = torch.zeros_like(updates[0][key], dtype=torch.float32)
        for update, weight in zip(updates, weights):
            accumulator += update[key].float() * float(weight)
        result[key] = accumulator
    assert_finite_update(result, "weighted aggregate")
    return result, weights


def rdp_epsilon(noise_multiplier, sample_rate, steps, delta):
    """Exact RDP composition for a full-participation Gaussian mechanism.

    The current research configuration uses sample_rate=1.0. Subsampled-client
    accounting is intentionally rejected rather than approximated silently.
    """
    if not math.isclose(sample_rate, 1.0, rel_tol=0.0, abs_tol=1e-12):
        raise NotImplementedError(
            "This notebook currently supports formal RDP accounting only for "
            "client_fraction=1.0. Keep full client participation or add a "
            "validated subsampled-Gaussian accountant."
        )
    if noise_multiplier <= 0:
        return math.inf
    orders = np.concatenate([
        np.arange(1.1, 10.0, 0.1),
        np.arange(10.0, 64.0, 1.0),
        np.asarray([64.0, 96.0, 128.0, 256.0]),
    ])
    rdp = steps * orders / (2.0 * noise_multiplier ** 2)
    epsilons = rdp + math.log(1.0 / delta) / (orders - 1.0)
    return float(np.min(epsilons))


def noise_multiplier_for_target(target_epsilon, sample_rate, steps, delta):
    if math.isinf(target_epsilon):
        return 0.0
    if target_epsilon <= 0:
        raise ValueError("target_epsilon must be positive")
    low, high = 1e-4, 1.0
    while rdp_epsilon(high, sample_rate, steps, delta) > target_epsilon:
        high *= 2.0
        if high > 1e6:
            raise RuntimeError("Unable to calibrate a finite Gaussian noise multiplier")
    for _ in range(100):
        mid = (low + high) / 2.0
        if rdp_epsilon(mid, sample_rate, steps, delta) > target_epsilon:
            low = mid
        else:
            high = mid
    return high


def add_secure_aggregate_noise(aggregate, normalized_weights, target_epsilon, sample_rate, steps, seed, max_effective_weight=None):
    if math.isinf(target_epsilon):
        return aggregate, {"noise_multiplier": 0.0, "epsilon": math.inf, "noise_std": 0.0}
    sigma = noise_multiplier_for_target(target_epsilon, sample_rate, steps, CFG.delta)
    # Replacement sensitivity of a capped weighted aggregate is bounded by 2*C*max(weight).
    effective_weight = float(np.max(normalized_weights)) if max_effective_weight is None else float(max_effective_weight)
    sensitivity = 2.0 * CFG.client_clip_norm * effective_weight
    std = sigma * sensitivity
    generator = torch.Generator(device="cpu").manual_seed(seed)
    private = OrderedDict()
    for key, value in aggregate.items():
        noise = torch.normal(0.0, std, size=value.shape, generator=generator, dtype=torch.float32)
        private[key] = value.float() + noise
    assert_finite_update(private, "private aggregate")
    return private, {"noise_multiplier": sigma, "epsilon": rdp_epsilon(sigma, sample_rate, steps, CFG.delta), "noise_std": std}


def apply_delta(global_state, delta):
    output = OrderedDict((k, v.detach().cpu().clone()) for k, v in global_state.items())
    for key, value in delta.items():
        output[key] = output[key].float() + value.float()
    if any(torch.is_floating_point(v) and not torch.isfinite(v).all() for v in output.values()):
        raise FloatingPointError("Updated global model is non-finite")
    return output


def update_bytes(update):
    return sum(v.numel() * v.element_size() for v in update.values())

In [26]:
# C3 Local training, AMA weights, anomaly filtering and PoS+BFT simulation
@dataclass
class ClientHistory:
    trust: float = 1.0
    previous_norm: float | None = None


def train_local(global_model, indices, seed, label_poison=False, fgsm_poison=False, learning_rate=None):
    seed_everything(seed)
    model = copy.deepcopy(global_model).to(DEVICE)
    lr = CFG.learning_rate if learning_rate is None else float(learning_rate)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=CFG.momentum, weight_decay=CFG.weight_decay)
    loader = DataLoader(Subset(train_aug, list(map(int, indices))), batch_size=CFG.local_batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=CUDA_USABLE, generator=torch.Generator().manual_seed(seed))
    losses = []
    started = time.perf_counter()
    model.train()
    for _ in range(CFG.local_epochs):
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=CUDA_USABLE)
            y = y.to(DEVICE, non_blocking=CUDA_USABLE)
            if label_poison:
                mask = torch.rand(len(y), device=DEVICE) < CFG.data_poison_fraction
                y = torch.where(mask, (y + 1) % 10, y)
            # Persistent malicious-client FGSM: perturb local training inputs.
            if fgsm_poison:
                x_seed = x.detach().clone().requires_grad_(True)
                seed_loss = F.cross_entropy(model(x_seed), y)
                grad = torch.autograd.grad(seed_loss, x_seed, retain_graph=False, create_graph=False)[0]
                mean_t = torch.tensor(MEAN, device=DEVICE, dtype=x.dtype).view(1,3,1,1)
                std_t = torch.tensor(STD, device=DEVICE, dtype=x.dtype).view(1,3,1,1)
                eps_t = (FGSM_EPSILON / std_t)
                lo = (torch.zeros_like(mean_t) - mean_t) / std_t
                hi = (torch.ones_like(mean_t) - mean_t) / std_t
                x = torch.maximum(torch.minimum(x_seed.detach() + eps_t * grad.sign(), hi), lo)

            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x), y)
            if not torch.isfinite(loss):
                raise FloatingPointError("Local loss became non-finite")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            optimizer.step()
            losses.append(float(loss.item()))
    assert_finite_model(model, "local model")
    return OrderedDict((k, v.detach().cpu().clone()) for k, v in model.state_dict().items()), {
        "loss": float(np.mean(losses)),
        "samples": len(indices),
        "seconds": time.perf_counter() - started,
    }


def ama_weight(samples, local_loss, norm, history):
    """
    Paper-aligned clean reconstruction.

    The paper explicitly gives sample-count weighting |Di|/sum|Dj| for the
    global update. It describes additional AMA factors (quality, diversity,
    representativeness, trust) but does not disclose an executable equation.

    To avoid inventing a strong undocumented weighting rule, clean trusted
    clients use sample-count weighting. Trust becomes active only when later
    robustness phases change a client's trust state.
    """
    return max(1e-9, float(samples) * float(history.trust))

def sampled_update_vector(update, maximum=4096):
    vector = np.concatenate([v.detach().cpu().float().reshape(-1).numpy() for v in update.values()])
    if len(vector) > maximum:
        step = max(1, len(vector) // maximum)
        vector = vector[::step][:maximum]
    return vector


def robust_acceptance(updates, enabled):
    if not enabled or len(updates) < 4:
        return np.ones(len(updates), dtype=bool)
    norms = np.asarray([update_norm(u) for u in updates])
    median = np.median(norms)
    mad = np.median(np.abs(norms - median)) + 1e-12
    norm_ok = np.abs(norms - median) / (1.4826 * mad) <= CFG.anomaly_mad_threshold
    vectors = np.vstack([sampled_update_vector(u) for u in updates])
    center = np.median(vectors, axis=0)
    center_norm = np.linalg.norm(center) + 1e-12
    cosine = (vectors @ center) / ((np.linalg.norm(vectors, axis=1) + 1e-12) * center_norm)
    cosine_ok = cosine > -0.10
    return norm_ok & cosine_ok


def pos_bft_verify(client_updates, trusts, update_hashes, seed):
    """Simulated PoS+BFT integrity validation over actual received updates."""
    client_ids = list(client_updates)
    if not client_ids:
        return set(), 0, 0.0, 0, 0

    started = time.perf_counter()
    rng = np.random.default_rng(seed)
    validator_count = min(5, len(client_ids))
    stake_values = np.asarray(
        [max(1e-9, trusts[client_id].trust) for client_id in client_ids],
        dtype=float,
    )
    validators = rng.choice(
        client_ids,
        size=validator_count,
        replace=False,
        p=stake_values / stake_values.sum(),
    ).tolist()

    byzantine_count = min(
        validator_count,
        int(round(
            validator_count * CFG.pos_bft_byzantine_fraction
        )),
    )
    byzantine_validators = set(
        rng.choice(
            validators,
            size=byzantine_count,
            replace=False,
        ).tolist()
    ) if byzantine_count else set()

    tolerated = (validator_count - 1) // 3
    threshold = 2 * tolerated + 1
    accepted = set()

    for client_id, update in client_updates.items():
        expected_digest = hash_update(update)
        received_digest = update_hashes.get(client_id, "")
        syntactically_valid = (
            len(received_digest) == 64
            and all(character in "0123456789abcdef" for character in received_digest)
        )
        integrity_valid = (
            syntactically_valid
            and expected_digest == received_digest
        )

        yes_votes = 0
        for validator in validators:
            honest_vote = integrity_valid
            vote = (
                not honest_vote
                if validator in byzantine_validators
                else honest_vote
            )
            yes_votes += int(vote)

        if yes_votes >= threshold:
            accepted.add(client_id)

    messages = 2 * validator_count * max(0, validator_count - 1)
    latency_ms = (
        (time.perf_counter() - started) * 1000
        + 3 * CFG.simulated_network_rtt_ms
        + messages * CFG.simulated_message_processing_ms
    )
    return (
        accepted,
        messages,
        latency_ms,
        validator_count,
        byzantine_count,
    )


def hash_update(update):
    h = hashlib.sha256()
    for key, value in update.items():
        h.update(key.encode())
        h.update(value.numpy().tobytes())
    return h.hexdigest()

In [27]:
# Best-checkpoint, cosine-LR, resumable EPP runner

def global_round_lr(round_index):
    frac=round_index/max(1,CFG.global_rounds-1)
    cosine=0.5*(1.0+math.cos(math.pi*frac))
    factor=CFG.min_learning_rate_factor+(1.0-CFG.min_learning_rate_factor)*cosine
    return CFG.learning_rate*factor

def run_epp_experiment(run_name,n_clients,target_epsilon,seed,attack="NONE",aggregation_mode="AMA"):
    progress(
        f"EPP RUN START → {run_name} | clients={n_clients} | "
        f"epsilon={target_epsilon} | attack={attack} | aggregation={aggregation_mode}"
    )
    seed_everything(seed)
    final_model_path=DIRS["models"]/f"{run_name}.pt"
    final_metrics_path=stage_csv(run_name)
    partial_path=stage_checkpoint(run_name)

    if final_model_path.exists() and final_metrics_path.exists() and not CFG.force_rerun:
        checkpoint=load_pipeline_checkpoint(final_model_path,map_location="cpu")
        validate_checkpoint_fingerprint(checkpoint,final_model_path.name)
        model=make_model()
        model.load_state_dict(checkpoint["state_dict"])
        existing_log=pd.read_csv(final_metrics_path)
        progress(
            f"EPP RUN REUSED → {run_name} | best_round={checkpoint.get('best_round')} | "
            f"logged_rounds={len(existing_log)}"
        )
        return model.cpu(),existing_log

    partitions=client_partitions(n_clients,seed)
    model=make_model()
    histories={cid:ClientHistory() for cid in partitions}

    logs=[]
    start_round=0
    best_state=None
    best_round=0
    best_val_acc=-math.inf
    best_val_loss=math.inf
    rounds_without_improvement=0

    if partial_path.exists() and not CFG.force_rerun:
        state=load_pipeline_checkpoint(partial_path,map_location="cpu")
        validate_checkpoint_fingerprint(state,partial_path.name)
        model.load_state_dict(state["model"])
        histories={int(k):ClientHistory(**v) for k,v in state["histories"].items()}
        logs=state.get("logs",[])
        start_round=int(state.get("next_round",0))
        best_state=state.get("best_state")
        best_round=int(state.get("best_round",0))
        best_val_acc=float(state.get("best_val_acc",-math.inf))
        best_val_loss=float(state.get("best_val_loss",math.inf))
        rounds_without_improvement=int(state.get("rounds_without_improvement",0))
        progress(
            f"EPP RESUME → {run_name} | next={start_round+1}/{CFG.global_rounds} | "
            f"best={best_val_acc:.4f}@{best_round}"
        )

    trainable=trainable_keys(model)
    sample_rate=CFG.client_fraction
    malicious_count=int(round(n_clients*CFG.adversarial_client_fraction))
    malicious_rng=np.random.default_rng(seed+700)
    malicious=set(
        malicious_rng.choice(n_clients,size=malicious_count,replace=False).tolist()
    ) if malicious_count else set()

    round_times=[]

    for round_index in range(start_round,CFG.global_rounds):
        round_started=time.perf_counter()
        current_lr=global_round_lr(round_index)
        progress(
            f"EPP ROUND START → {run_name} | {round_index+1}/{CFG.global_rounds} | lr={current_lr:.6f}"
        )

        round_rng=np.random.default_rng(seed+round_index*100003)
        selected_count=max(1,math.ceil(n_clients*CFG.client_fraction))
        selected=round_rng.choice(n_clients,size=selected_count,replace=False).tolist()
        global_state=OrderedDict(
            (k,v.detach().cpu().clone()) for k,v in model.state_dict().items()
        )

        updates,weights,metadata,hashes=[],[],[],{}
        client_progress_step=max(1,len(selected)//4)

        for client_position,client_id in enumerate(selected,start=1):
            local_state,info=train_local(
                model,
                partitions[client_id],
                seed+round_index*10000+client_id,
                label_poison=client_id in malicious and attack=="DATA_POISON",
                fgsm_poison=client_id in malicious and attack=="FGSM_TRAIN",
                learning_rate=current_lr,
            )
            raw_update=state_delta(local_state,global_state,trainable)

            if client_id in malicious and attack=="MODEL_POISON":
                raw_update=OrderedDict(
                    (k,v*CFG.model_poison_scale) for k,v in raw_update.items()
                )

            original_norm=update_norm(raw_update)
            should_clip=(not math.isinf(target_epsilon)) or CFG.clip_clean_updates

            if should_clip:
                update,_=clip_update(raw_update,CFG.client_clip_norm)
                was_clipped=original_norm>CFG.client_clip_norm
            else:
                update=raw_update
                was_clipped=False

            if aggregation_mode=="FEDAVG":
                weight=float(info["samples"])
            elif aggregation_mode=="AMA":
                weight=ama_weight(
                    info["samples"],info["loss"],update_norm(update),histories[client_id]
                )
            else:
                raise ValueError(f"Unknown aggregation_mode={aggregation_mode}")

            updates.append(update)
            weights.append(weight)
            hashes[client_id]=hash_update(update)
            metadata.append({
                "client_id":client_id,
                "malicious":client_id in malicious,
                "original_norm":original_norm,
                "was_clipped":was_clipped,
                **info,
            })

            if (
                client_position==1
                or client_position%client_progress_step==0
                or client_position==len(selected)
            ):
                progress(
                    f"  local clients → {client_position}/{len(selected)} "
                    f"for {run_name} round {round_index+1}"
                )

        detection_started=time.perf_counter()
        accepted_mask=robust_acceptance(updates,enabled=attack!="NONE")
        detection_seconds=time.perf_counter()-detection_started

        accepted_updates=[u for u,ok in zip(updates,accepted_mask) if ok]
        accepted_weights=[w for w,ok in zip(weights,accepted_mask) if ok]
        accepted_client_ids=[
            m["client_id"] for m,ok in zip(metadata,accepted_mask) if ok
        ]
        accepted_update_map={
            cid:update for cid,update in zip(accepted_client_ids,accepted_updates)
        }

        (
            verified,consensus_messages,consensus_latency_ms,
            consensus_validators,consensus_byzantine,
        )=pos_bft_verify(
            accepted_update_map,
            histories,
            {cid:hashes[cid] for cid in accepted_client_ids},
            seed+round_index,
        )

        positions=[
            j for j,cid in enumerate(accepted_client_ids)
            if cid in verified
        ]
        if not positions:
            raise RuntimeError(
                f"{run_name}: no client updates accepted at round {round_index+1}"
            )

        accepted_updates=[accepted_updates[j] for j in positions]
        accepted_weights=[accepted_weights[j] for j in positions]
        accepted_client_ids=[accepted_client_ids[j] for j in positions]

        aggregate,normalized_weights=weighted_average(
            accepted_updates,accepted_weights
        )
        aggregate,privacy=add_secure_aggregate_noise(
            aggregate,normalized_weights,target_epsilon,sample_rate,
            CFG.global_rounds,seed+900000+round_index,
        )
        model.load_state_dict(apply_delta(global_state,aggregate),strict=True)
        assert_finite_model(model,f"{run_name} round {round_index+1}")

        for client_id,item in zip(selected,metadata):
            histories[client_id].previous_norm=item["original_norm"]
            if client_id in accepted_client_ids:
                histories[client_id].trust=min(2.0,histories[client_id].trust*1.01)
            else:
                histories[client_id].trust=max(0.05,histories[client_id].trust*0.85)

        validation=evaluate(model,VALSET)

        improved=(
            validation["accuracy"]>best_val_acc+CFG.best_min_delta
            or (
                abs(validation["accuracy"]-best_val_acc)<=CFG.best_min_delta
                and validation["loss"]<best_val_loss
            )
        )

        if improved:
            best_val_acc=float(validation["accuracy"])
            best_val_loss=float(validation["loss"])
            best_round=round_index+1
            best_state=OrderedDict(
                (k,v.detach().cpu().clone()) for k,v in model.state_dict().items()
            )
            rounds_without_improvement=0
            progress(
                f"  ★ NEW BEST → round={best_round} "
                f"val_acc={best_val_acc:.4f} val_loss={best_val_loss:.4f}"
            )
        else:
            rounds_without_improvement+=1

        true_malicious=np.asarray([m["malicious"] for m in metadata],dtype=bool)
        rejected=~accepted_mask
        tp=int(np.sum(rejected&true_malicious))
        fp=int(np.sum(rejected&~true_malicious))
        fn=int(np.sum(~rejected&true_malicious))

        logs.append({
            "run_name":run_name,
            "system":"EPP_BCFL_RECONSTRUCTION_V4",
            "aggregation_mode":aggregation_mode,
            "seed":seed,
            "num_clients":n_clients,
            "round":round_index+1,
            "learning_rate":current_lr,
            "target_epsilon":target_epsilon,
            "accounted_epsilon":privacy["epsilon"],
            "noise_multiplier":privacy["noise_multiplier"],
            "attack":attack,
            "validation_loss":validation["loss"],
            "validation_accuracy":validation["accuracy"],
            "validation_f1":validation["f1"],
            "is_best":bool(improved),
            "best_round_so_far":best_round,
            "best_val_accuracy_so_far":best_val_acc,
            "selected_clients":len(selected),
            "accepted_clients":len(accepted_client_ids),
            "mean_raw_update_norm":float(np.mean([m["original_norm"] for m in metadata])),
            "clipping_fraction":float(np.mean([bool(m["was_clipped"]) for m in metadata])),
            "communication_bytes":sum(update_bytes(u) for u in accepted_updates),
            "consensus_messages":consensus_messages,
            "consensus_latency_ms":consensus_latency_ms,
            "consensus_validators":consensus_validators,
            "consensus_byzantine":consensus_byzantine,
            "consensus_energy_j":blockchain_energy(
                sum(update_bytes(u) for u in accepted_updates),
                consensus_messages,consensus_latency_ms,
            ),
            "detection_seconds":detection_seconds,
            "security_response_seconds":detection_seconds + consensus_latency_ms/1000.0,
            "malicious_detection_recall":tp/(tp+fn) if tp+fn else np.nan,
            "false_positive_rate":fp/max(1,np.sum(~true_malicious)),
        })

        round_seconds=time.perf_counter()-round_started
        round_times.append(round_seconds)
        eta_seconds=float(np.mean(round_times))*(CFG.global_rounds-(round_index+1))

        progress(
            f"EPP ROUND DONE → {run_name} | {round_index+1}/{CFG.global_rounds} | "
            f"val_acc={validation['accuracy']:.4f} | val_f1={validation['f1']:.4f} | "
            f"best={best_val_acc:.4f}@{best_round} | no_improve={rounds_without_improvement} | "
            f"round={round_seconds:.1f}s | ETA≈{eta_seconds/60:.1f} min"
        )

        torch.save({
            "model":OrderedDict(
                (k,v.detach().cpu().clone()) for k,v in model.state_dict().items()
            ),
            "best_state":best_state,
            "best_round":best_round,
            "best_val_acc":best_val_acc,
            "best_val_loss":best_val_loss,
            "rounds_without_improvement":rounds_without_improvement,
            "histories":{cid:asdict(value) for cid,value in histories.items()},
            "logs":logs,
            "next_round":round_index+1,
            "fingerprint":EXPERIMENT_FINGERPRINT,
        },partial_path)

        pd.DataFrame(logs).to_csv(final_metrics_path,index=False)
        maybe_write_resume_bundle(round_index+1)

        if (
            round_index+1>=CFG.early_stop_min_rounds
            and rounds_without_improvement>=CFG.early_stop_patience
        ):
            progress(
                f"EARLY STOP → {run_name} at round {round_index+1}; "
                f"restoring best round {best_round}"
            )
            break

    if best_state is None:
        raise RuntimeError(f"{run_name}: no valid best checkpoint was produced")

    model.load_state_dict(best_state,strict=True)
    final_state=OrderedDict(
        (k,v.detach().cpu().clone()) for k,v in model.state_dict().items()
    )
    torch.save({
        "state_dict":final_state,
        "run_name":run_name,
        "best_round":best_round,
        "best_val_accuracy":best_val_acc,
        "best_val_loss":best_val_loss,
        "config":asdict(CFG),
        "fingerprint":EXPERIMENT_FINGERPRINT,
    },final_model_path)

    if partial_path.exists():
        partial_path.unlink()

    progress(
        f"EPP RUN COMPLETE → {run_name} | BEST round={best_round} "
        f"val_acc={best_val_acc:.4f}"
    )
    return model.cpu(),pd.DataFrame(logs)


In [28]:
# B9 Secure-aggregation ring arithmetic validation
# This checks modular mask cancellation and communication overhead.
# It remains a simulation, not production cryptographic SMPC.

def ring_secure_aggregate_vectors(vectors, seed, scale=1_000_000):
    if not vectors:
        raise ValueError("At least one vector is required.")

    arrays = [np.asarray(vector, dtype=np.float64) for vector in vectors]
    if any(array.shape != arrays[0].shape for array in arrays):
        raise ValueError(
            "All secure-aggregation vectors must have the same shape."
        )
    if not all(np.isfinite(array).all() for array in arrays):
        raise FloatingPointError(
            "Secure-aggregation input contains NaN/Inf."
        )

    modulus = np.int64(2**61 - 1)
    quantized = [
        np.mod(
            np.rint(array * scale).astype(np.int64),
            modulus,
        )
        for array in arrays
    ]

    rng = np.random.default_rng(seed)
    masks = [
        rng.integers(
            0,
            modulus,
            size=arrays[0].shape,
            dtype=np.int64,
        )
        for _ in arrays
    ]

    shares = []
    for index, quantized_vector in enumerate(quantized):
        previous_mask = masks[index - 1]
        own_mask = masks[index]
        # Each intermediate remains within signed int64 range.
        share = np.mod(
            quantized_vector + own_mask - previous_mask,
            modulus,
        )
        shares.append(share)

    # Do not use np.sum(..., dtype=int64): summing many modular shares can
    # overflow before the final modulus is applied.
    recovered_q = np.zeros_like(shares[0], dtype=np.int64)
    for share in shares:
        recovered_q = np.mod(
            recovered_q + share,
            modulus,
        )

    signed = np.where(
        recovered_q > modulus // 2,
        recovered_q - modulus,
        recovered_q,
    )
    recovered = signed.astype(np.float64) / scale
    expected = np.sum(arrays, axis=0)

    # Each independently rounded client contributes at most 0.5/scale error.
    tolerance = len(arrays) / (2 * scale) + 1e-12
    maximum_error = float(
        np.max(np.abs(recovered - expected))
    )

    return {
        "max_abs_error": maximum_error,
        "quantization_tolerance": tolerance,
        "recovered": bool(
            np.allclose(
                recovered,
                expected,
                atol=tolerance,
                rtol=0.0,
            )
        ),
        "input_bytes": int(
            sum(array.nbytes for array in arrays)
        ),
        "share_bytes": int(
            sum(share.nbytes for share in shares)
        ),
        "clients": len(arrays),
        "elements": arrays[0].size,
    }


secure_demo_vectors = [
    np.random.default_rng(CFG.seed + client).normal(size=4096)
    for client in range(
        max(4, min(10, CFG.client_grid[0]))
    )
]
SECURE_AGGREGATION_AUDIT = pd.DataFrame([
    ring_secure_aggregate_vectors(
        secure_demo_vectors,
        CFG.seed,
    )
])
SECURE_AGGREGATION_AUDIT.to_csv(
    DIRS["metrics"] / "secure_aggregation_audit.csv",
    index=False,
)

if not bool(SECURE_AGGREGATION_AUDIT.iloc[0]["recovered"]):
    raise AssertionError(
        "Ring secure-aggregation arithmetic validation failed."
    )

display(SECURE_AGGREGATION_AUDIT)

,max_abs_error,quantization_tolerance,recovered,input_bytes,share_bytes,clients,elements
0,0.000003,0.000005,True,327680,327680,10,4096


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [29]:
def final_row(model,log,run_name,extra=None):
    if log.empty:
        raise RuntimeError(f'No log rows for {run_name}')
    best_round=int(log['best_round_so_far'].iloc[-1]) if 'best_round_so_far' in log.columns else int(log.loc[log['validation_accuracy'].idxmax(),'round'])
    best_log=log[log['round']==best_round].iloc[-1].to_dict()
    test=evaluate(model,TESTSET)
    best_log.update({
        'best_round':best_round,
        'test_accuracy':test['accuracy'],
        'test_precision':test['precision'],
        'test_recall':test['recall'],
        'test_f1':test['f1'],
        'model_path':str(DIRS['models']/f'{run_name}.pt'),
    })
    if extra:
        best_log.update(extra)
    return best_log

def append_summary(filename,rows):
    path=DIRS['metrics']/filename
    old=pd.read_csv(path) if path.exists() else pd.DataFrame()
    new=pd.DataFrame(rows)
    merged=pd.concat([old,new],ignore_index=True) if not old.empty else new
    if not merged.empty:
        merged=merged.drop_duplicates(['run_name'],keep='last')
    merged.to_csv(path,index=False)
    return merged


In [30]:

def completed_run_names():
    """Return run names already completed in the imported/current resume state."""
    names = set()

    # Completed model files are the strongest signal.
    if DIRS["models"].exists():
        for p in DIRS["models"].glob("*.pt"):
            names.add(p.stem)

    # Summary CSVs are an additional signal.
    for csv_name in (
        "epp_clean_all_summary.csv",
        "fedavg_all_summary.csv",
        "epp_privacy_all_summary.csv",
        "epp_attack_all_summary.csv",
        "epp_fgsm_all_summary.csv",
    ):
        p = DIRS["metrics"] / csv_name
        if p.exists():
            try:
                df = pd.read_csv(p)
                if "run_name" in df.columns:
                    names.update(df["run_name"].dropna().astype(str).tolist())
            except Exception:
                pass

    return names


def pending_jobs_for_phase(phase, jobs):
    """
    Filter the full job list so already-completed jobs from the resume bundle are skipped.
    """
    completed = completed_run_names()
    pending = []

    if phase in ("CLEAN", "FEDAVG"):
        prefix = "epp_clean" if phase == "CLEAN" else "fedavg"
        for n, seed in jobs:
            name = f"{prefix}_n{n}_seed{seed}"
            if name not in completed:
                pending.append((n, seed))

    elif phase == "PRIVACY":
        for n, eps, seed in jobs:
            en = str(eps).replace(".", "p")
            name = f"epp_privacy_n{n}_eps{en}_seed{seed}"
            if name not in completed:
                pending.append((n, eps, seed))

    elif phase == "ATTACKS":
        for attack, n, seed in jobs:
            name = f"epp_{attack.lower()}_n{n}_seed{seed}"
            if name not in completed:
                pending.append((attack, n, seed))

    elif phase == "FGSM":
        # FGSM outputs are CSV-only, so check summary run_name.
        for n, seed in jobs:
            name = f"epp_fgsm_n{n}_seed{seed}"
            if name not in completed:
                pending.append((n, seed))

    else:
        pending = list(jobs)

    return completed, pending


In [31]:

# B12–B18 — Run all required EPP reconstruction experiments sequentially
# Each run auto-resumes from its own Drive checkpoint.

def append_summary_unique(filename, row, keys=('run_name',)):
    path=DIRS['metrics']/filename
    old=pd.read_csv(path) if path.exists() else pd.DataFrame()
    merged=pd.concat([old,pd.DataFrame([row])],ignore_index=True)
    if all(k in merged.columns for k in keys):
        merged=merged.drop_duplicates(list(keys),keep='last')
    merged.to_csv(path,index=False)
    return merged

def finish_epp_run(name, n, epsilon, seed, attack='NONE', extra=None):
    model,log=run_epp_experiment(
        name,n,epsilon,seed,attack=attack,aggregation_mode='AMA'
    )
    row=final_row(model,log,name,extra or {})
    if 'security_response_seconds' in log.columns:
        row['mean_security_response_seconds']=float(log['security_response_seconds'].mean())
        row['p95_security_response_seconds']=float(log['security_response_seconds'].quantile(.95))
    if 'detection_seconds' in log.columns:
        row['mean_detection_seconds']=float(log['detection_seconds'].mean())
    write_resume_bundle()
    return model,log,row

n=REPRESENTATIVE_CLIENTS
seed=SEEDS[0]

# 1) CLEAN
clean_name=f'epp_clean_n{n}_seed{seed}'
clean_model,clean_log,clean_row=finish_epp_run(
    clean_name,n,math.inf,seed,attack='NONE',
    extra={'scenario':'CLEAN'}
)
append_summary_unique('epp_clean_all_summary.csv',clean_row)

# 2) DP epsilon=0.5
for eps in TARGET_EPSILONS:
    en=str(eps).replace('.','p')
    name=f'epp_privacy_n{n}_eps{en}_seed{seed}'
    model,log,row=finish_epp_run(
        name,n,eps,seed,attack='NONE',
        extra={'scenario':'PRIVACY','epsilon':eps}
    )
    append_summary_unique('epp_privacy_all_summary.csv',row)

# 3) DATA POISON
name=f'epp_data_poison_n{n}_seed{seed}'
model,log,row=finish_epp_run(
    name,n,math.inf,seed,attack='DATA_POISON',
    extra={'scenario':'DATA_POISON','attack_type':'DATA_POISON'}
)
append_summary_unique('epp_attack_all_summary.csv',row)

# 4) MODEL POISON
name=f'epp_model_poison_n{n}_seed{seed}'
model,log,row=finish_epp_run(
    name,n,math.inf,seed,attack='MODEL_POISON',
    extra={'scenario':'MODEL_POISON','attack_type':'MODEL_POISON'}
)
append_summary_unique('epp_attack_all_summary.csv',row)

# 5) Persistent malicious-client FGSM training (20% malicious clients, every FL round)
name=f'epp_fgsm_train_n{n}_seed{seed}'
fgsm_train_model,fgsm_train_log,row=finish_epp_run(
    name,n,math.inf,seed,attack='FGSM_TRAIN',
    extra={'scenario':'FGSM_TRAIN','attack_type':'FGSM_TRAIN',
           'adversarial_client_fraction':ADVERSARIAL_CLIENT_FRACTION,
           'fgsm_epsilon':FGSM_EPSILON}
)
append_summary_unique('epp_attack_all_summary.csv',row)

# 6) Supplementary raw white-box FGSM on the CLEAN checkpoint
clean_state=load_pipeline_checkpoint(DIRS['models']/f'{clean_name}.pt')
validate_checkpoint_fingerprint(clean_state,f'{clean_name}.pt')
whitebox_model=make_model()
whitebox_model.load_state_dict(clean_state['state_dict'])
whitebox=evaluate(whitebox_model,TESTSET,adversarial_epsilon=FGSM_EPSILON)
whitebox_row={
    'run_name':f'epp_fgsm_whitebox_n{n}_seed{seed}',
    'scenario':'FGSM_WHITEBOX_SUPPLEMENTARY',
    'num_clients':n,
    'seed':seed,
    'attack_type':'FGSM_WHITEBOX',
    'fgsm_epsilon':FGSM_EPSILON,
    'test_accuracy':whitebox['accuracy'],
    'test_f1':whitebox['f1'],
}
append_summary_unique('epp_fgsm_all_summary.csv',whitebox_row)

print('✅ EPP required reconstruction runs complete/reused.',flush=True)


[07:16:06] EPP RUN START → epp_clean_n10_seed42 | clients=10 | epsilon=inf | attack=NONE | aggregation=AMA
[07:16:06] EPP ROUND START → epp_clean_n10_seed42 | 1/50 | lr=0.050000
[07:16:18]   local clients → 1/10 for epp_clean_n10_seed42 round 1
[07:16:26]   local clients → 2/10 for epp_clean_n10_seed42 round 1
[07:16:43]   local clients → 4/10 for epp_clean_n10_seed42 round 1
[07:17:02]   local clients → 6/10 for epp_clean_n10_seed42 round 1
[07:17:21]   local clients → 8/10 for epp_clean_n10_seed42 round 1
[07:17:37]   local clients → 10/10 for epp_clean_n10_seed42 round 1
[07:17:40]   ★ NEW BEST → round=1 val_acc=0.1200 val_loss=2.6416
[07:17:40] EPP ROUND DONE → epp_clean_n10_seed42 | 1/50 | val_acc=0.1200 | val_f1=0.0380 | best=0.1200@1 | no_improve=0 | round=93.1s | ETA≈76.0 min
[07:17:40] EPP ROUND START → epp_clean_n10_seed42 | 2/50 | lr=0.049950


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[07:17:49]   local clients → 1/10 for epp_clean_n10_seed42 round 2
[07:17:59]   local clients → 2/10 for epp_clean_n10_seed42 round 2
[07:18:15]   local clients → 4/10 for epp_clean_n10_seed42 round 2
[07:18:33]   local clients → 6/10 for epp_clean_n10_seed42 round 2
[07:18:52]   local clients → 8/10 for epp_clean_n10_seed42 round 2
[07:19:12]   local clients → 10/10 for epp_clean_n10_seed42 round 2
[07:19:14]   ★ NEW BEST → round=2 val_acc=0.2468 val_loss=2.0285
[07:19:14] EPP ROUND DONE → epp_clean_n10_seed42 | 2/50 | val_acc=0.2468 | val_f1=0.1685 | best=0.2468@2 | no_improve=0 | round=94.5s | ETA≈75.0 min
[07:19:14] EPP ROUND START → epp_clean_n10_seed42 | 3/50 | lr=0.049799


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[07:19:24]   local clients → 1/10 for epp_clean_n10_seed42 round 3
[07:19:33]   local clients → 2/10 for epp_clean_n10_seed42 round 3
[07:19:52]   local clients → 4/10 for epp_clean_n10_seed42 round 3
[07:20:12]   local clients → 6/10 for epp_clean_n10_seed42 round 3
[07:20:30]   local clients → 8/10 for epp_clean_n10_seed42 round 3
[07:20:46]   local clients → 10/10 for epp_clean_n10_seed42 round 3
[07:20:48] EPP ROUND DONE → epp_clean_n10_seed42 | 3/50 | val_acc=0.2076 | val_f1=0.1415 | best=0.2468@2 | no_improve=1 | round=94.2s | ETA≈73.6 min
[07:20:48] EPP ROUND START → epp_clean_n10_seed42 | 4/50 | lr=0.049548


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


... (458 outputs left out)

[13:20:56] RESUME SNAPSHOT → /content/drive/MyDrive/LSB_EHEALTH_COMPLETE_BASE_PAPERS_COLAB/EPP_BCFL_REPRO_V4_Resume.zip | 58.4 MB
[13:20:56] EPP RUN COMPLETE → epp_fgsm_train_n10_seed42 | BEST round=47 val_acc=0.8030
[13:21:05] CHECKPOINT LOAD → epp_clean_n10_seed42.pt


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ EPP required reconstruction runs complete/reused.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:

# B19 — Device/network heterogeneity reconstruction
hr=[]
for n in CLIENT_GRID:
    for seed in SEEDS:
        rng=np.random.default_rng(seed+n)
        model_bytes=int(sum(p.numel()*p.element_size() for p in make_model().parameters()))
        for cid in range(n):
            bw=float(rng.uniform(1,20))
            cf=float(rng.uniform(.5,2))
            battery=float(rng.uniform(.2,1))
            comm=(model_bytes*8)/(bw*1_000_000)
            train_s=1/cf
            hr.append({
                'num_clients':n,'seed':seed,'client_id':cid,
                'bandwidth_mbps':bw,'compute_factor':cf,'battery':battery,
                'communication_seconds':comm,'normalized_training_seconds':train_s,
                'estimated_energy_j':comm*.8+train_s*1.5,
            })
heterogeneity=pd.DataFrame(hr)
heterogeneity.to_csv(DIRS['metrics']/ 'epp_device_heterogeneity.csv',index=False)
display(heterogeneity.describe())


,num_clients,seed,client_id,bandwidth_mbps,compute_factor,battery,communication_seconds,normalized_training_seconds,estimated_energy_j
count,10.0,10.0,10.00000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,10.0,42.0,4.50000,9.687333,1.082352,0.674061,20.077128,1.089582,17.696076
std,0.0,0.0,3.02765,6.490335,0.433404,0.213568,23.460172,0.482033,18.627616
min,10.0,42.0,0.00000,1.121401,0.523756,0.395002,4.633964,0.542294,4.520613
25%,10.0,42.0,2.25000,3.864570,0.658900,0.540161,5.977873,0.739545,6.730679
50%,10.0,42.0,4.50000,9.956306,1.176093,0.607210,9.011497,0.852355,9.601513
75%,10.0,42.0,6.75000,14.827594,1.352364,0.820734,24.255101,1.533294,21.490023
max,10.0,42.0,9.00000,18.982556,1.844017,0.986230,78.441572,1.909286,63.884959


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [33]:

# B20 — Reproduced EPP benchmark summary (primary comparison source)
frames=[]
for filename in (
    'epp_clean_all_summary.csv',
    'epp_privacy_all_summary.csv',
    'epp_attack_all_summary.csv',
    'epp_fgsm_all_summary.csv',
):
    p=DIRS['metrics']/filename
    if p.exists():
        df=pd.read_csv(p)
        df['source_file']=filename
        frames.append(df)

epp_benchmark=pd.concat(frames,ignore_index=True,sort=False) if frames else pd.DataFrame()
epp_benchmark.to_csv(DIRS['metrics']/ 'epp_reproduced_benchmark.csv',index=False)

show_cols=[c for c in [
    'run_name','scenario','attack_type','epsilon','num_clients','seed',
    'test_accuracy','test_f1','best_round',
    'mean_security_response_seconds','p95_security_response_seconds'
] if c in epp_benchmark.columns]
display(epp_benchmark[show_cols] if show_cols else epp_benchmark)

print('✅ This measured table — not the published headline table — is the primary EPP comparison source.',flush=True)


,run_name,scenario,attack_type,epsilon,num_clients,seed,test_accuracy,test_f1,best_round,mean_security_response_seconds,p95_security_response_seconds
0,epp_clean_n10_seed42,CLEAN,NaN,NaN,10,42,0.8066,0.803886,50.0,0.390337,0.589279
1,epp_privacy_n10_eps0p5_seed42,PRIVACY,NaN,0.5,10,42,0.1096,0.036618,18.0,0.382217,0.595076
2,epp_data_poison_n10_seed42,DATA_POISON,DATA_POISON,NaN,10,42,0.7941,0.791290,47.0,0.437399,0.660908
3,epp_model_poison_n10_seed42,MODEL_POISON,MODEL_POISON,NaN,10,42,0.7832,0.776746,40.0,0.373253,0.539825
4,epp_fgsm_train_n10_seed42,FGSM_TRAIN,FGSM_TRAIN,NaN,10,42,0.7897,0.784468,47.0,0.413401,0.620947
5,epp_fgsm_whitebox_n10_seed42,FGSM_WHITEBOX_SUPPLEMENTARY,FGSM_WHITEBOX,NaN,10,42,0.0237,0.021254,NaN,NaN,NaN


✅ This measured table — not the published headline table — is the primary EPP comparison source.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Final export

The final Drive folder contains separate DBH and EPP reconstruction artifacts, checkpoints, manifests, measured summaries and context-only published-reference tables.

**Do not delete the Drive folder after completion.** It is the baseline evidence package used for later LSB comparison.


In [34]:

# Final — package all base-paper evidence
final_manifest={
    'project':'LSB_EHEALTH_COMPLETE_BASE_PAPERS_COLAB',
    'comparison_rule':'LSB superiority is judged against measured reconstructed base-paper outputs under the available compute environment.',
    'dbh_root':str(MASTER_ROOT/'DBH_CHAIN_REPRO_V1'),
    'epp_root':str(MASTER_ROOT/'EPP_BCFL_REPRO_V4'),
    'epp_primary_configuration':{
        'clients':REPRESENTATIVE_CLIENTS,
        'seed':SEEDS[0],
        'rounds':GLOBAL_ROUNDS,
        'local_epochs':LOCAL_EPOCHS,
        'classes_per_client':CLASSES_PER_CLIENT,
        'privacy_epsilon':0.5,
        'malicious_client_fraction':ADVERSARIAL_CLIENT_FRACTION,
    },
    'updated_at_epoch':time.time(),
}
(MASTER_ROOT/'BASE_PAPERS_FINAL_MANIFEST.json').write_text(
    json.dumps(final_manifest,indent=2,default=str)
)

drive_archive=MASTER_ROOT/'LSB_EHEALTH_COMPLETE_BASE_PAPERS_RESULTS_SLIM.zip'
tmp_archive=Path('/content/LSB_EHEALTH_COMPLETE_BASE_PAPERS_RESULTS_SLIM.zip')
if tmp_archive.exists():
    tmp_archive.unlink()
if drive_archive.exists():
    drive_archive.unlink()

with zipfile.ZipFile(tmp_archive,'w',zipfile.ZIP_DEFLATED) as z:
    manifest_file=MASTER_ROOT/'BASE_PAPERS_FINAL_MANIFEST.json'
    if manifest_file.exists():
        z.write(manifest_file,manifest_file.relative_to(MASTER_ROOT))

    for subroot in (
        MASTER_ROOT/'DBH_CHAIN_REPRO_V1',
        MASTER_ROOT/'EPP_BCFL_REPRO_V4',
    ):
        for folder_name in ('metrics','manifests','logs','plots'):
            folder=subroot/folder_name
            if folder.exists():
                for f in folder.rglob('*'):
                    if f.is_file():
                        z.write(f,f.relative_to(MASTER_ROOT))

    # Include final EPP models needed for later attack/comparison validation.
    model_dir=MASTER_ROOT/'EPP_BCFL_REPRO_V4'/'models'
    if model_dir.exists():
        for f in model_dir.glob('*.pt'):
            z.write(f,f.relative_to(MASTER_ROOT))

shutil.copy2(tmp_archive,drive_archive)

print('\n✅ COMPLETE BASE-PAPER COLAB PIPELINE FINISHED',flush=True)
print('Drive root:',MASTER_ROOT,flush=True)
print('Final slim Drive ZIP:',drive_archive,flush=True)
print('DBH measured benchmark:',MASTER_ROOT/'DBH_CHAIN_REPRO_V1/metrics/dbh_reproduced_benchmark.csv',flush=True)
print('EPP measured benchmark:',MASTER_ROOT/'EPP_BCFL_REPRO_V4/metrics/epp_reproduced_benchmark.csv',flush=True)



✅ COMPLETE BASE-PAPER COLAB PIPELINE FINISHED
Drive root: /content/drive/MyDrive/LSB_EHEALTH_COMPLETE_BASE_PAPERS_COLAB
Final slim Drive ZIP: /content/drive/MyDrive/LSB_EHEALTH_COMPLETE_BASE_PAPERS_COLAB/LSB_EHEALTH_COMPLETE_BASE_PAPERS_RESULTS_SLIM.zip
DBH measured benchmark: /content/drive/MyDrive/LSB_EHEALTH_COMPLETE_BASE_PAPERS_COLAB/DBH_CHAIN_REPRO_V1/metrics/dbh_reproduced_benchmark.csv
EPP measured benchmark: /content/drive/MyDrive/LSB_EHEALTH_COMPLETE_BASE_PAPERS_COLAB/EPP_BCFL_REPRO_V4/metrics/epp_reproduced_benchmark.csv
